# Flipkart Traffic Demand Prediction
**Goal:** R² > 0.95 on held-out test set  
**Target:** `demand` — continuous, range [0, 1], heavily right-skewed  

---
| Phase | Description |
|-------|-------------|
| 1 | Setup & Imports |
| 2 | Data Loading |
| 3 | Exploratory Data Analysis |
| 4 | Feature Engineering |
| 5 | Baseline Model (LightGBM) |
| 6 | Hyperparameter Tuning (Optuna) |
| 7 | Full Ensemble (LGB + XGB + CatBoost) |
| 8 | SHAP Feature Importance |
| 9 | Generate Submission |

---
## Phase 1 — Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import pygeohash as pgh
import shap

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='muted')

SEED  = 42
np.random.seed(SEED)

BASE  = '/Users/sudarshansudhakar/Downloads/FlipKart Challenge/dataset'
OUT   = '/Users/sudarshansudhakar/Downloads/FlipKart Challenge/traffic_prediction'

os.makedirs(f'{OUT}/data',        exist_ok=True)
os.makedirs(f'{OUT}/models',      exist_ok=True)
os.makedirs(f'{OUT}/submissions', exist_ok=True)
os.makedirs(f'{OUT}/features',    exist_ok=True)

print('All imports OK')

---
## Phase 2 — Data Loading

In [ ]:
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
sub   = pd.read_csv(f'{BASE}/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'Sub   : {sub.shape}')
print(f'\nSub columns: {sub.columns.tolist()}')

---
## Phase 3 — Exploratory Data Analysis

In [ ]:
# --- 3.1 dtypes & shape
print('=== TRAIN dtypes ===')
print(train.dtypes)
print('\n=== TEST dtypes ===')
print(test.dtypes)

In [ ]:
# --- 3.2 head
train.head(10)

In [ ]:
test.head(10)

In [ ]:
# --- 3.3 describe
train.describe(include='all')

In [ ]:
test.describe(include='all')

In [ ]:
# --- 3.4 Null counts
null_df = pd.DataFrame({
    'train_nulls'   : train.isnull().sum(),
    'train_null_pct': (train.isnull().sum() / len(train) * 100).round(2),
    'test_nulls'    : test.isnull().sum(),
    'test_null_pct' : (test.isnull().sum() / len(test) * 100).round(2),
})
print(null_df)

In [ ]:
# --- 3.5 Categorical unique values
for col in ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']:
    for name, df in [('TRAIN', train), ('TEST', test)]:
        vals = sorted(df[col].dropna().unique().tolist())
        print(f'[{name}] {col} ({df[col].nunique()} unique): {vals}')
    print()

In [ ]:
# --- 3.6 Demand distribution stats
d = train['demand'].dropna()
print('=== DEMAND DISTRIBUTION ===')
print(f'  mean      : {d.mean():.6f}')
print(f'  median    : {d.median():.6f}')
print(f'  std       : {d.std():.6f}')
print(f'  skewness  : {skew(d):.6f}')
print(f'  kurtosis  : {kurtosis(d):.6f}')
print(f'  min       : {d.min():.6f}')
print(f'  max       : {d.max():.6f}')
for p in [1, 5, 25, 75, 95, 99]:
    print(f'  p{p:<3}      : {np.percentile(d, p):.6f}')

In [ ]:
# --- 3.7 Demand plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(d, bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('Demand (raw)')
axes[0].set_xlabel('demand')

axes[1].hist(np.log1p(d), bins=100, color='coral', edgecolor='none')
axes[1].set_title('Demand (log1p)')
axes[1].set_xlabel('log1p(demand)')

axes[2].boxplot(d, vert=True, patch_artist=True, boxprops=dict(facecolor='lightgreen'))
axes[2].set_title('Demand Boxplot')

plt.tight_layout()
plt.savefig(f'{OUT}/data/demand_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.8 Demand by categoricals
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flatten(), ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']):
    order = train.groupby(col)['demand'].median().sort_values(ascending=False).index
    sns.boxplot(data=train, x=col, y='demand', order=order, ax=ax, palette='Set2')
    ax.set_title(f'Demand by {col}')
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f'{OUT}/data/demand_by_categoricals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3.9 Demand by hour
train['_hour'] = train['timestamp'].str.split(':').str[0].astype(int)
hourly = train.groupby('_hour')['demand'].mean()

plt.figure(figsize=(14, 4))
plt.plot(hourly.index, hourly.values, marker='o', color='steelblue')
plt.title('Average Demand by Hour')
plt.xlabel('Hour')
plt.ylabel('Mean Demand')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/data/demand_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
train.drop(columns=['_hour'], inplace=True)

In [ ]:
# --- 3.10 Geohash overlap
print(f'Train geohashes : {train["geohash"].nunique()}')
print(f'Test  geohashes : {test["geohash"].nunique()}')
print(f'Common          : {len(set(train["geohash"]) & set(test["geohash"]))}')
print(f'Test-only       : {len(set(test["geohash"]) - set(train["geohash"]))}')

# Geohash prefix coverage
for n in [1, 2, 3, 4, 5]:
    tr = set(train['geohash'].str[:n])
    te = set(test['geohash'].str[:n])
    print(f'  prefix-{n} unique in train={len(tr)}, test={len(te)}, common={len(tr & te)}')

In [ ]:
# --- 3.11 Save data_summary.txt
d = train['demand'].dropna()
summary = [
    '=== SHAPES ===',
    f'Train: {train.shape}  Test: {test.shape}',
    '\n=== TRAIN dtypes ===',
    train.dtypes.to_string(),
    '\n=== NULL COUNTS (train) ===',
    train.isnull().sum().to_string(),
    '\n=== NULL COUNTS (test) ===',
    test.isnull().sum().to_string(),
    '\n=== DEMAND STATS ===',
    f'mean={d.mean():.6f}  median={d.median():.6f}  std={d.std():.6f}',
    f'skew={skew(d):.4f}  kurtosis={kurtosis(d):.4f}',
    f'min={d.min():.6f}  max={d.max():.6f}',
]
with open(f'{OUT}/data_summary.txt', 'w') as f:
    f.write('\n'.join(summary))
print('data_summary.txt saved.')

---
## Phase 3.5 — Deep Exploratory Data Analysis
Covers: Temporal · Geospatial · Feature Correlation · Combined Patterns

In [ ]:
# Working copy — never mutates train / test
eda = train.copy()

PLOT_DIR = OUT + '/notebooks'
os.makedirs(PLOT_DIR, exist_ok=True)
print('Deep-EDA working copy ready:', eda.shape)

### A — Temporal Analysis

In [ ]:
# A1 — Timestamp format detection & parsing
print('Sample timestamp values:', eda['timestamp'].unique()[:15].tolist())
print('dtype :', eda['timestamp'].dtype)
print('Total unique timestamps:', eda['timestamp'].nunique())

# Format is 'H:MM' (hour 0-23, minute 0/15/30/45)
eda['hour']      = eda['timestamp'].str.split(':').str[0].astype(int)
eda['minute']    = eda['timestamp'].str.split(':').str[1].astype(int)
eda['time_slot'] = eda['hour'] * 4 + eda['minute'] // 15   # 0..95
eda['time_dec']  = eda['hour'] + eda['minute'] / 60        # decimal hour

print('\nHour range    :', eda['hour'].min(), '→', eda['hour'].max())
print('Minute values :', sorted(eda['minute'].unique()))
print('Time-slots    :', eda['time_slot'].nunique(), '(expected 96 for 15-min grid)')
print('Day values    :', sorted(eda['day'].unique()), '→ numeric (likely week-of-year/day-ID)')

In [ ]:
# A2 — Demand by hour of day
hourly = eda.groupby('hour')['demand'].agg(['mean','median','std']).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hourly['hour'], hourly['mean'],   marker='o', lw=2, label='Mean',   color='steelblue')
ax.plot(hourly['hour'], hourly['median'], marker='s', lw=2, label='Median', color='coral')
ax.fill_between(hourly['hour'],
                hourly['mean'] - hourly['std'],
                hourly['mean'] + hourly['std'],
                alpha=0.15, color='steelblue', label='±1 std')
ax.set_title('Demand by Hour of Day', fontsize=14)
ax.set_xlabel('Hour')
ax.set_ylabel('Demand')
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/A2_demand_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print(hourly.to_string(index=False))

In [ ]:
# A3 — Demand by day
print('Day column analysis:')
print('  unique values:', sorted(eda['day'].unique()))
print('  dtype        :', eda['day'].dtype)
print('  value counts :')
print(eda['day'].value_counts().sort_index())
print('  NOTE: day=48 → train only, day=49 → test only (literal split boundary)')

daily = eda.groupby('day')['demand'].agg(['mean','median','std','count']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(daily['day'].astype(str), daily['mean'], color=['steelblue','coral'][:len(daily)])
axes[0].set_title('Mean Demand by Day')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Mean Demand')

# 15-min time-slot demand aggregated by day
for day_val, grp in eda.groupby('day'):
    slot_mean = grp.groupby('time_slot')['demand'].mean()
    axes[1].plot(slot_mean.index, slot_mean.values, label=f'Day {day_val}', lw=1.5)
axes[1].set_title('Mean Demand per 15-min slot by Day')
axes[1].set_xlabel('Time slot (0=00:00, 95=23:45)')
axes[1].set_ylabel('Mean Demand')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/A3_demand_by_day.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# A4 — Peak vs off-peak hours
hourly_mean = eda.groupby('hour')['demand'].mean().sort_values(ascending=False)
overall_mean = eda['demand'].mean()

peak_threshold    = overall_mean * 1.25
offpeak_threshold = overall_mean * 0.75

peak_hours    = hourly_mean[hourly_mean >= peak_threshold].index.sort_values().tolist()
offpeak_hours = hourly_mean[hourly_mean <= offpeak_threshold].index.sort_values().tolist()

print(f'Overall mean demand   : {overall_mean:.4f}')
print(f'Peak threshold (×1.25): {peak_threshold:.4f}')
print(f'Peak hours            : {peak_hours}')
print(f'Off-peak threshold    : {offpeak_threshold:.4f}')
print(f'Off-peak hours        : {offpeak_hours}')

hourly_sorted = eda.groupby('hour')['demand'].mean().reset_index()
colors = []
for h in hourly_sorted['hour']:
    if h in peak_hours:
        colors.append('#e74c3c')
    elif h in offpeak_hours:
        colors.append('#2ecc71')
    else:
        colors.append('#3498db')

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(hourly_sorted['hour'], hourly_sorted['demand'], color=colors)
ax.axhline(peak_threshold,    color='red',   linestyle='--', label='Peak threshold')
ax.axhline(offpeak_threshold, color='green', linestyle='--', label='Off-peak threshold')
ax.set_title('Peak (red) vs Off-Peak (green) Hours')
ax.set_xlabel('Hour')
ax.set_ylabel('Mean Demand')
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/A4_peak_hours.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# A5 — Heatmap: day × hour with mean demand
pivot_dh = eda.pivot_table(values='demand', index='day', columns='hour', aggfunc='mean')

fig, ax = plt.subplots(figsize=(18, 3))
sns.heatmap(pivot_dh, ax=ax, cmap='YlOrRd', annot=False,
            linewidths=0.3, cbar_kws={'label': 'Mean Demand'})
ax.set_title('Mean Demand Heatmap: Day × Hour')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/A5_heatmap_day_hour.png', dpi=150, bbox_inches='tight')
plt.show()

# Also: 15-min resolution heatmap
pivot_slot = eda.pivot_table(values='demand', index='day', columns='time_slot', aggfunc='mean')
fig, ax = plt.subplots(figsize=(24, 3))
sns.heatmap(pivot_slot, ax=ax, cmap='YlOrRd', annot=False, linewidths=0,
            cbar_kws={'label': 'Mean Demand'})
ax.set_title('Mean Demand Heatmap: Day × 15-min Slot')
ax.set_xlabel('Time Slot (0=00:00 → 95=23:45)')
ax.set_ylabel('Day')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/A5_heatmap_day_timeslot.png', dpi=150, bbox_inches='tight')
plt.show()

### B — Geospatial Analysis

In [ ]:
# B6 — Decode geohash → lat / lon
import pygeohash as pgh

geo_coords = {}
for gh in eda['geohash'].unique():
    try:
        lat, lon, _, _ = pgh.decode_exactly(gh)
        geo_coords[gh] = (lat, lon)
    except:
        geo_coords[gh] = (np.nan, np.nan)

eda['lat'] = eda['geohash'].map(lambda g: geo_coords[g][0])
eda['lon'] = eda['geohash'].map(lambda g: geo_coords[g][1])

print(f'Decoded {len(geo_coords)} unique geohashes')
print(f'Lat range: {eda["lat"].min():.4f} → {eda["lat"].max():.4f}')
print(f'Lon range: {eda["lon"].min():.4f} → {eda["lon"].max():.4f}')
print(eda[['geohash','lat','lon']].drop_duplicates().head(5))

In [ ]:
# B7 — Scatter lat/lon colored by mean demand
geo_demand = eda.groupby(['geohash','lat','lon'])['demand'].mean().reset_index()
geo_demand.columns = ['geohash','lat','lon','mean_demand']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc = axes[0].scatter(geo_demand['lon'], geo_demand['lat'],
                     c=geo_demand['mean_demand'], cmap='YlOrRd',
                     s=30, alpha=0.8, edgecolors='none')
plt.colorbar(sc, ax=axes[0], label='Mean Demand')
axes[0].set_title('Mean Demand by Location (all geohashes)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

# Hexbin version for density
hb = axes[1].hexbin(geo_demand['lon'], geo_demand['lat'],
                    C=geo_demand['mean_demand'], gridsize=40,
                    cmap='YlOrRd', reduce_C_function=np.mean)
plt.colorbar(hb, ax=axes[1], label='Mean Demand')
axes[1].set_title('Hexbin: Mean Demand by Location')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/B7_geospatial_demand.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# B8 — Top 10 highest demand geohash locations
geo_stats_full = eda.groupby('geohash')['demand'].agg(
    mean='mean', median='median', std='std', count='count'
).reset_index().sort_values('mean', ascending=False)
geo_stats_full['lat'] = geo_stats_full['geohash'].map(lambda g: geo_coords[g][0])
geo_stats_full['lon'] = geo_stats_full['geohash'].map(lambda g: geo_coords[g][1])

top10_high = geo_stats_full.head(10)
print('=== Top 10 HIGHEST demand geohashes ===')
print(top10_high[['geohash','lat','lon','mean','median','std','count']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10_high['geohash'], top10_high['mean'], color='#e74c3c')
ax.set_title('Top 10 Geohashes by Mean Demand')
ax.set_xlabel('Mean Demand')
for bar, val in zip(bars, top10_high['mean']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/B8_top10_high_demand_geo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# B9 — Top 10 lowest demand geohash locations
top10_low = geo_stats_full[geo_stats_full['count'] >= 10].tail(10).sort_values('mean')
print('=== Top 10 LOWEST demand geohashes (min 10 obs) ===')
print(top10_low[['geohash','lat','lon','mean','median','std','count']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top10_low['geohash'], top10_low['mean'], color='#2ecc71')
ax.set_title('Top 10 Geohashes by Lowest Mean Demand (≥10 obs)')
ax.set_xlabel('Mean Demand')
for bar, val in zip(bars, top10_low['mean']):
    ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/B9_top10_low_demand_geo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# B10 — Unique geohashes: train vs test overlap
train_geo = set(train['geohash'])
test_geo  = set(test['geohash'])
common    = train_geo & test_geo
test_only = test_geo - train_geo
train_only= train_geo - test_geo

print(f'Train unique geohashes : {len(train_geo)}')
print(f'Test  unique geohashes : {len(test_geo)}')
print(f'Common (seen in both)  : {len(common)}  ({100*len(common)/len(test_geo):.1f}% of test)')
print(f'Test-only (UNSEEN)     : {len(test_only)}  ({100*len(test_only)/len(test_geo):.1f}% of test)')
print(f'Train-only             : {len(train_only)}')
if test_only:
    print(f'\nSample unseen test geohashes: {list(test_only)[:10]}')

# Prefix coverage
print('\nPrefix-level coverage:')
for n in [1,2,3,4,5]:
    tr_p = set(train['geohash'].str[:n])
    te_p = set(test['geohash'].str[:n])
    miss = te_p - tr_p
    print(f'  prefix-{n}: train={len(tr_p):4d}  test={len(te_p):4d}  '
          f'common={len(tr_p & te_p):4d}  test-only={len(miss)}')

# Venn-style bar
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(['Geohashes'], [len(common)],    label=f'Common ({len(common)})',    color='#3498db')
ax.barh(['Geohashes'], [len(test_only)], label=f'Test-only ({len(test_only)})', color='#e74c3c',
        left=[len(common)])
ax.barh(['Geohashes'], [len(train_only)],label=f'Train-only ({len(train_only)})', color='#2ecc71',
        left=[len(common)+len(test_only)])
ax.set_title('Geohash Overlap: Train vs Test')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/B10_geohash_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

### C — Feature Correlation

In [ ]:
# C11 — Full correlation heatmap (encoded categoricals)
corr_df = eda.copy()
corr_df['RoadType_enc']      = corr_df['RoadType'].map({'Residential':0,'Street':1,'Highway':2})
corr_df['LargeVehicles_enc'] = corr_df['LargeVehicles'].map({'Not Allowed':0,'Allowed':1})
corr_df['Landmarks_enc']     = corr_df['Landmarks'].map({'No':0,'Yes':1})
corr_df['Weather_enc']       = corr_df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3})

num_cols = ['demand','hour','minute','time_slot','day','lat','lon',
             'RoadType_enc','NumberofLanes','LargeVehicles_enc',
             'Landmarks_enc','Temperature','Weather_enc']
corr_matrix = corr_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Full Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C11_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Demand correlations sorted
print('\nCorrelation with demand (sorted):')
print(corr_matrix['demand'].drop('demand').sort_values(key=abs, ascending=False).to_string())

In [ ]:
# C12 — Demand distribution by RoadType
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

road_types = sorted(eda['RoadType'].dropna().unique())
colors_rt  = ['#3498db', '#e74c3c', '#2ecc71']

# KDE
for rt, col in zip(road_types, colors_rt):
    subset = eda[eda['RoadType'] == rt]['demand']
    subset.plot.kde(ax=axes[0], label=f'{rt} (n={len(subset):,})', color=col, lw=2)
axes[0].set_title('Demand KDE by RoadType')
axes[0].set_xlabel('Demand')
axes[0].legend(fontsize=8)
axes[0].set_xlim(-0.05, 0.8)

# Box
order_rt = eda.groupby('RoadType')['demand'].median().sort_values(ascending=False).index
sns.boxplot(data=eda, x='RoadType', y='demand', order=order_rt,
            palette='Set2', ax=axes[1])
axes[1].set_title('Demand Boxplot by RoadType')

# Stats table
rt_stats = eda.groupby('RoadType')['demand'].agg(['mean','median','std','count'])
axes[2].axis('off')
tbl = axes[2].table(cellText=rt_stats.round(4).values,
                    rowLabels=rt_stats.index,
                    colLabels=['Mean','Median','Std','Count'],
                    cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)
axes[2].set_title('Stats by RoadType', pad=20)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C12_demand_by_roadtype.png', dpi=150, bbox_inches='tight')
plt.show()
print(rt_stats)

In [ ]:
# C13 — Demand distribution by Weather
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

weather_types = sorted(eda['Weather'].dropna().unique())
pal_w = {'Sunny':'#f39c12','Rainy':'#3498db','Foggy':'#95a5a6','Snowy':'#1abc9c'}

for wt in weather_types:
    subset = eda[eda['Weather'] == wt]['demand']
    subset.plot.kde(ax=axes[0], label=f'{wt} (n={len(subset):,})',
                   color=pal_w.get(wt,'grey'), lw=2)
axes[0].set_title('Demand KDE by Weather')
axes[0].set_xlabel('Demand')
axes[0].legend(fontsize=9)
axes[0].set_xlim(-0.05, 0.8)

order_w = eda.groupby('Weather')['demand'].median().sort_values(ascending=False).index
sns.violinplot(data=eda, x='Weather', y='demand', order=order_w,
               palette=pal_w, ax=axes[1], cut=0)
axes[1].set_title('Demand Violin by Weather')

w_stats = eda.groupby('Weather')['demand'].agg(['mean','median','std','count'])
axes[2].axis('off')
tbl = axes[2].table(cellText=w_stats.round(4).values,
                    rowLabels=w_stats.index,
                    colLabels=['Mean','Median','Std','Count'],
                    cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 1.8)
axes[2].set_title('Stats by Weather', pad=20)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C13_demand_by_weather.png', dpi=150, bbox_inches='tight')
plt.show()
print(w_stats)

In [ ]:
# C14 — Demand vs Temperature (scatter + trend)
from scipy.stats import pearsonr, spearmanr

temp_data = eda[['Temperature','demand']].dropna()
pr, pp = pearsonr(temp_data['Temperature'],  temp_data['demand'])
sr, sp = spearmanr(temp_data['Temperature'], temp_data['demand'])
print(f'Pearson  r={pr:.4f}  p={pp:.2e}')
print(f'Spearman r={sr:.4f}  p={sp:.2e}')

# Bin temperature and compute mean demand
temp_data = temp_data.copy()
temp_data['temp_bin'] = pd.cut(temp_data['Temperature'], bins=20)
bin_stats = temp_data.groupby('temp_bin', observed=True)['demand'].agg(['mean','count'])
bin_centers = [iv.mid for iv in bin_stats.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sample = temp_data.sample(min(8000, len(temp_data)), random_state=42)
axes[0].scatter(sample['Temperature'], sample['demand'],
                alpha=0.15, s=5, color='steelblue')

# Polynomial trend
z = np.polyfit(temp_data['Temperature'], temp_data['demand'], 2)
p = np.poly1d(z)
xs = np.linspace(temp_data['Temperature'].min(), temp_data['Temperature'].max(), 200)
axes[0].plot(xs, p(xs), color='red', lw=2, label=f'Poly-2 trend  Pearson r={pr:.3f}')
axes[0].set_title('Demand vs Temperature')
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Demand')
axes[0].legend()

axes[1].bar(bin_centers, bin_stats['mean'], width=2.5, color='coral', edgecolor='white')
axes[1].set_title('Mean Demand by Temperature Bin')
axes[1].set_xlabel('Temperature (bin center)')
axes[1].set_ylabel('Mean Demand')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C14_demand_vs_temperature.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C15 — Demand: LargeVehicles Allowed vs Not Allowed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lv in ['Allowed','Not Allowed']:
    subset = eda[eda['LargeVehicles'] == lv]['demand']
    subset.plot.kde(ax=axes[0], label=f'{lv} (n={len(subset):,})', lw=2)
axes[0].set_title('Demand KDE: LargeVehicles')
axes[0].set_xlabel('Demand')
axes[0].legend()
axes[0].set_xlim(-0.05, 0.8)

sns.boxplot(data=eda, x='LargeVehicles', y='demand',
            order=['Not Allowed','Allowed'], palette='Set1', ax=axes[1])
axes[1].set_title('Demand Boxplot: LargeVehicles')

lv_stats = eda.groupby('LargeVehicles')['demand'].agg(['mean','median','std','count'])
print(lv_stats)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C15_demand_largevehicles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# C16 — Demand: Landmarks Yes vs No
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lm in ['Yes','No']:
    subset = eda[eda['Landmarks'] == lm]['demand']
    subset.plot.kde(ax=axes[0], label=f'Landmarks={lm} (n={len(subset):,})', lw=2)
axes[0].set_title('Demand KDE: Landmarks')
axes[0].set_xlabel('Demand')
axes[0].legend()
axes[0].set_xlim(-0.05, 0.8)

sns.boxplot(data=eda, x='Landmarks', y='demand',
            order=['No','Yes'], palette='Set2', ax=axes[1])
axes[1].set_title('Demand Boxplot: Landmarks')

lm_stats = eda.groupby('Landmarks')['demand'].agg(['mean','median','std','count'])
print(lm_stats)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/C16_demand_landmarks.png', dpi=150, bbox_inches='tight')
plt.show()

### D — Combined Patterns

In [ ]:
# D17 — Demand by RoadType × Hour (grouped line chart)
road_hour = eda.groupby(['RoadType','hour'])['demand'].mean().reset_index()
road_hour = road_hour.dropna(subset=['RoadType'])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

colors_rt = {'Residential':'#3498db','Street':'#e74c3c','Highway':'#2ecc71'}
for rt, grp in road_hour.groupby('RoadType'):
    axes[0].plot(grp['hour'], grp['demand'],
                 marker='o', lw=2, label=rt, color=colors_rt.get(rt,'grey'))
axes[0].set_title('Mean Demand by RoadType × Hour')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Mean Demand')
axes[0].set_xticks(range(0, 24))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Pivot heatmap: RoadType × Hour
pivot_rh = road_hour.pivot(index='RoadType', columns='hour', values='demand')
sns.heatmap(pivot_rh, ax=axes[1], cmap='YlOrRd', annot=True, fmt='.3f',
            linewidths=0.5, cbar_kws={'label':'Mean Demand'},
            annot_kws={'size': 7})
axes[1].set_title('RoadType × Hour Heatmap')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/D17_roadtype_x_hour.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# D18 — Demand heatmap: Weather × RoadType
wr_pivot = eda.groupby(['Weather','RoadType'])['demand'].mean().unstack()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(wr_pivot, ax=axes[0], cmap='YlOrRd', annot=True, fmt='.4f',
            linewidths=0.5, cbar_kws={'label':'Mean Demand'})
axes[0].set_title('Mean Demand: Weather × RoadType')

# Count heatmap (sample sizes)
wr_count = eda.groupby(['Weather','RoadType'])['demand'].count().unstack()
sns.heatmap(wr_count, ax=axes[1], cmap='Blues', annot=True, fmt='d',
            linewidths=0.5, cbar_kws={'label':'Count'})
axes[1].set_title('Sample Counts: Weather × RoadType')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/D18_weather_x_roadtype.png', dpi=150, bbox_inches='tight')
plt.show()
print(wr_pivot)

In [ ]:
# D19 — Are road features static per geohash? (stability check)
geo_road_check = eda.groupby('geohash').agg(
    RoadType_nunique      = ('RoadType',       'nunique'),
    Lanes_nunique         = ('NumberofLanes',  'nunique'),
    LargeVehicles_nunique = ('LargeVehicles',  'nunique'),
    Landmarks_nunique     = ('Landmarks',      'nunique'),
    obs_count             = ('demand',         'count')
).reset_index()

print('=== Road Feature Stability per Geohash ===')
for col in ['RoadType_nunique','Lanes_nunique','LargeVehicles_nunique','Landmarks_nunique']:
    vc = geo_road_check[col].value_counts().sort_index()
    pct1 = 100 * (geo_road_check[col] == 1).mean()
    print(f'\n{col}: {pct1:.1f}% of geohashes have exactly 1 unique value')
    print(vc.to_string())

# Show unstable geohashes
unstable = geo_road_check[
    (geo_road_check['RoadType_nunique'] > 1) |
    (geo_road_check['Lanes_nunique'] > 1)
]
print(f'\nGeohashes with changing RoadType or Lanes: {len(unstable)}')
if len(unstable) > 0:
    print(unstable.sort_values('obs_count', ascending=False).head(10).to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
feat_cols = ['RoadType_nunique','Lanes_nunique','LargeVehicles_nunique','Landmarks_nunique']
feat_labels = ['RoadType','NumberofLanes','LargeVehicles','Landmarks']
for ax, col, label in zip(axes, feat_cols, feat_labels):
    vc = geo_road_check[col].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color='steelblue')
    ax.set_title(f'{label}\n# unique values per geohash')
    ax.set_xlabel('# unique values')
    ax.set_ylabel('# geohashes')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/D19_road_feature_stability.png', dpi=150, bbox_inches='tight')
plt.show()

### Summary of Key EDA Insights

In [ ]:
# Final insight summary — top 10 findings for feature engineering

# Compute supporting numbers on the fly
geo_road_check2 = eda.groupby('geohash').agg(
    RoadType_nunique = ('RoadType','nunique'),
    Lanes_nunique    = ('NumberofLanes','nunique'),
).reset_index()
pct_stable_road  = 100*(geo_road_check2['RoadType_nunique']  == 1).mean()
pct_stable_lanes = 100*(geo_road_check2['Lanes_nunique'] == 1).mean()

hourly_mean2 = eda.groupby('hour')['demand'].mean()
peak_h   = hourly_mean2.idxmax()
trough_h = hourly_mean2.idxmin()
peak_ratio = hourly_mean2.max() / hourly_mean2.min()

rt_means = eda.groupby('RoadType')['demand'].mean().sort_values(ascending=False)
w_means  = eda.groupby('Weather')['demand'].mean().sort_values(ascending=False)
lv_means = eda.groupby('LargeVehicles')['demand'].mean()
lm_means = eda.groupby('Landmarks')['demand'].mean()

train_geo2 = set(train['geohash'])
test_geo2  = set(test['geohash'])
pct_unseen = 100*len(test_geo2 - train_geo2)/len(test_geo2)

from scipy.stats import pearsonr as pr2
temp_nonan = eda[['Temperature','demand']].dropna()
temp_corr, _ = pr2(temp_nonan['Temperature'], temp_nonan['demand'])

lv_ratio = lv_means.get('Allowed', np.nan) / lv_means.get('Not Allowed', np.nan)
lm_ratio = lm_means.get('Yes', np.nan) / lm_means.get('No', np.nan)

print('=' * 72)
print('TOP 10 EDA INSIGHTS FOR FEATURE ENGINEERING')
print('=' * 72)

insights = [
    (1, 'STRONG TIME SIGNAL',
     f'Demand varies {peak_ratio:.1f}x across hours (peak h={peak_h}, '
     f'trough h={trough_h}). Hour + 15-min time_slot are the #1 predictors. '
     f'Use cyclical sin/cos encoding to preserve continuity.'),

    (2, 'GEOHASH IS THE DOMINANT FEATURE',
     f'Individual geohash explains most demand variance. '
     f'Mean demand per geohash ranges from near-0 to 1.0. '
     f'Target-encode geohash: geo_mean, geo_median, geo_std are critical features.'),

    (3, 'GEO x TIME INTERACTION',
     f'Different locations peak at different hours. '
     f'geo_slot_mean (geohash × 15-min slot) captures local temporal pattern '
     f'and will be one of the strongest engineered features.'),

    (4, 'ROAD FEATURES ARE NEAR-STATIC PER LOCATION',
     f'{pct_stable_road:.1f}% of geohashes have a single RoadType; '
     f'{pct_stable_lanes:.1f}% have a single NumberofLanes. '
     f'NaN RoadType can be imputed from geohash mode, recovering ~600 rows cleanly.'),

    (5, 'HIGHWAY >> STREET >> RESIDENTIAL DEMAND',
     f'Mean demand: {rt_means.to_dict()}. '
     f'Highway has substantially higher demand. '
     f'RoadType is a strong ordinal feature (encode as 0/1/2).'),

    (6, 'LARGE VEHICLES = HIGHER DEMAND ROADS',
     f'Allowed roads have {lv_ratio:.2f}x mean demand of Not-Allowed. '
     f'Strongly correlated with RoadType (Highways allow large vehicles). '
     f'road_capacity = NumberofLanes × (1+LargeVehicles_enc) is a useful proxy.'),

    (7, 'LANDMARKS NEAR = HIGHER DEMAND',
     f'Landmarks=Yes has {lm_ratio:.2f}x mean demand vs No. '
     f'Landmark presence indicates commercial/tourist zones. '
     f'Binary encode directly; consider geo×landmark interactions.'),

    (8, f'TEMPERATURE HAS WEAK BUT NON-LINEAR EFFECT (r={temp_corr:.3f})',
     f'Linear correlation is low ({temp_corr:.3f}) but binned analysis shows '
     f'extreme temperatures (very cold/hot) suppress demand. '
     f'Add temp² and absolute-temp features. Impute ~2,495 NaN with geohash median.'),

    (9, 'WEATHER HAS MODEST IMPACT',
     f'Weather means: {w_means.round(4).to_dict()}. '
     f'Differences are real but small. Snowy/Foggy reduce demand slightly. '
     f'Weather × Hour interaction may capture weather-dependent commute patterns.'),

    (10, f'{pct_unseen:.1f}% TEST GEOHASHES ARE UNSEEN',
     f'{len(test_geo2-train_geo2)} geohashes in test never appear in train. '
     f'Fallback to coarser prefix (geo3/geo4) mean demand for these cold-start locations. '
     f'Lat/lon features provide smooth spatial interpolation for unseen geohashes.'),
]

for idx, title, detail in insights:
    print(f'\n{idx:2d}. [{title}]')
    # Wrap text at 72 chars
    words = detail.split()
    line  = '    '
    for w in words:
        if len(line) + len(w) + 1 > 74:
            print(line)
            line = '    ' + w + ' '
        else:
            line += w + ' '
    print(line)

print('\n' + '=' * 72)
print('RECOMMENDED FEATURE PRIORITY ORDER:')
print('  1. geo_slot_mean  (geohash × 15-min slot target-encoded mean)')
print('  2. geo_hour_mean  (geohash × hour mean)')
print('  3. geo_mean       (overall geohash mean demand)')
print('  4. time_slot      (0..95 — 15-min resolution)')
print('  5. hour_sin/cos   (cyclical time encoding)')
print('  6. RoadType_enc   (ordinal: Residential=0, Street=1, Highway=2)')
print('  7. road_capacity  (NumberofLanes × LargeVehicles factor)')
print('  8. lat / lon      (raw coordinates from geohash decode)')
print('  9. geo3_mean      (prefix-3 region mean for cold-start geohashes)')
print(' 10. temp features  (Temperature, temp², temp_abs, temp_bin)')
print('=' * 72)

---
## Phase 4 — Comprehensive Feature Engineering
Pipeline fits **only on train**, transforms both train & test. Unseen geohashes fall back to global mean.

| Group | Features |
|-------|----------|
| 1 — Temporal | hour/minute, cyclical sin/cos, peak/night flags, time-of-day bucket, day cyclical, weekend |
| 2 — Geospatial | lat/lon, geohash-5/-4 truncation, distance-from-center, KMeans clusters (k chosen by silhouette) |
| 3 — Target Encoding | **5-fold OOF** mean/median/std/max across 10 key/interaction groupings (leak-safe) |
| 4 — Road / Infra | binary encodings, capacity & infrastructure scores, ordinal RoadType, lanes×roadtype |
| 5 — Weather / Env | demand-ordered Weather ordinal, quantile temp bins, severity score, interactions |
| 6 — Neighbor / Context | 8-neighbor mean/max demand, neighbor ratio |
| 7 — Frequency | geohash, geohash×hour, weather, roadtype frequency |

Outputs: `/features/train_features.csv`, `/features/test_features.csv`

In [ ]:
# 4.0 — Fresh working copies + shared OOF fold assignment
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

tr = train.copy()
te = test.copy()
TARGET = 'demand'

# Folds reused later by the model CV (same seed -> identical splits -> leak-free target encoding)
FE_FOLDS = 5
fe_kf = KFold(n_splits=FE_FOLDS, shuffle=True, random_state=SEED)
tr['_fold'] = -1
for f, (_, val_idx) in enumerate(fe_kf.split(tr)):
    tr.loc[tr.index[val_idx], '_fold'] = f
print('Fold sizes:', tr['_fold'].value_counts().sort_index().tolist())

In [ ]:
# ══ GROUP 1 — TEMPORAL FEATURES ══
def time_of_day_bucket(h):
    if 5 <= h <= 11:  return 0   # morning
    if 12 <= h <= 16: return 1   # afternoon
    if 17 <= h <= 20: return 2   # evening
    return 3                     # night

for df in [tr, te]:
    parts = df['timestamp'].str.split(':')
    df['hour']   = parts.str[0].astype(int)
    df['minute'] = parts.str[1].astype(int)
    # NOTE: timestamp is 'H:MM' — no seconds component available
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    # 2) cyclical encodings
    df['hour_sin']   = np.sin(2 * np.pi * df['hour']   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour']   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    # 3) peak-hour flag (7-9am, 5-8pm)
    df['is_peak_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)
    # 4) night flag (11pm-5am)
    df['is_night'] = df['hour'].isin([23, 0, 1, 2, 3, 4, 5]).astype(int)
    # 5) time-of-day bucket
    df['time_of_day'] = df['hour'].map(time_of_day_bucket)
    # 6) day cyclical (day is an absolute index; %7 -> pseudo day-of-week)
    df['day_of_week'] = df['day'] % 7
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    # 7) weekend flag
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print('Group 1 done. day unique -> train:', sorted(tr['day'].unique()),
      ' test:', sorted(te['day'].unique()))
print('  (day is constant within each split -> day-derived cols are zero-variance in train,')
print('   they are auto-dropped in the assembly step below.)')

In [ ]:
# ══ GROUP 2 — GEOSPATIAL FEATURES ══
import pygeohash as pgh

# 8) decode geohash -> lat / lon
all_geo = pd.unique(pd.concat([tr['geohash'], te['geohash']]))
coords = {}
for gh in all_geo:
    d = pgh.decode_exactly(gh)
    coords[gh] = (d.latitude, d.longitude)

for df in [tr, te]:
    df['lat'] = df['geohash'].map(lambda g: coords[g][0])
    df['lon'] = df['geohash'].map(lambda g: coords[g][1])
    # 9) / 10) coarser geohash truncations
    df['geohash5'] = df['geohash'].str[:5]
    df['geohash4'] = df['geohash'].str[:4]

# 11) haversine distance from city center (train mean lat/lon as proxy)
center_lat = tr['lat'].mean()
center_lon = tr['lon'].mean()
def haversine(lat, lon, clat, clon):
    R = 6371.0; p = np.pi / 180
    a = (0.5 - np.cos((clat - lat) * p) / 2
         + np.cos(lat * p) * np.cos(clat * p) * (1 - np.cos((clon - lon) * p)) / 2)
    return 2 * R * np.arcsin(np.sqrt(a))
for df in [tr, te]:
    df['dist_from_center'] = haversine(df['lat'].values, df['lon'].values, center_lat, center_lon)

# 12) KMeans on unique geohash coords; pick k by silhouette
geo_xy = pd.DataFrame([(g, coords[g][0], coords[g][1]) for g in all_geo],
                      columns=['geohash', 'lat', 'lon'])
best_k, best_sil, best_km = None, -1, None
for k in [20, 50, 100]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(geo_xy[['lat', 'lon']])
    sil = silhouette_score(geo_xy[['lat', 'lon']], labels)
    print(f'  k={k:3d}  silhouette={sil:.4f}')
    if sil > best_sil:
        best_k, best_sil, best_km = k, sil, km
print(f'-> chosen k={best_k} (silhouette={best_sil:.4f})')

# 13) assign cluster label per geohash
geo_xy['geo_cluster'] = best_km.predict(geo_xy[['lat', 'lon']])
cluster_map = dict(zip(geo_xy['geohash'], geo_xy['geo_cluster']))
for df in [tr, te]:
    df['geo_cluster'] = df['geohash'].map(cluster_map).astype(int)

# label-encode the truncated geohashes to integer codes (fit on combined categories)
for col in ['geohash5', 'geohash4']:
    cats = pd.Categorical(pd.concat([tr[col], te[col]]))
    mapping = {c: i for i, c in enumerate(cats.categories)}
    tr[col + '_enc'] = tr[col].map(mapping).astype(int)
    te[col + '_enc'] = te[col].map(mapping).astype(int)
print('Group 2 done.')

In [ ]:
# ══ GROUP 4 — ROAD & INFRASTRUCTURE (run before target encoding) ══
# Impute RoadType from geohash mode (EDA: 79.6% geohashes have a single RoadType)
def geohash_mode_fill(col):
    src = train.dropna(subset=[col])
    mode_by_geo = src.groupby('geohash')[col].agg(lambda s: s.mode().iat[0])
    global_mode = train[col].mode().iat[0]
    for df in [tr, te]:
        df[col] = df[col].fillna(df['geohash'].map(mode_by_geo)).fillna(global_mode)

geohash_mode_fill('RoadType')
geohash_mode_fill('Weather')

# 24) binary encodings
for df in [tr, te]:
    df['LargeVehicles_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
    df['Landmarks_bin']     = (df['Landmarks'] == 'Yes').astype(int)
    # 25) capacity / 26) infrastructure scores
    df['road_capacity_score']   = df['NumberofLanes'] * (1 + df['LargeVehicles_bin'])
    df['infrastructure_score']  = df['road_capacity_score'] + df['Landmarks_bin']

# 27) RoadType ordinal — natural ordering exists (EDA: Highway >> Street >> Residential),
#     derive the order data-driven from train mean demand
road_order = tr.groupby('RoadType')[TARGET].mean().sort_values().index.tolist()
road_map = {r: i for i, r in enumerate(road_order)}
print('RoadType ordinal (by ascending mean demand):', road_map)
for df in [tr, te]:
    df['RoadType_ord'] = df['RoadType'].map(road_map).astype(int)
    # 28) lanes x roadtype interaction
    df['lanes_x_roadtype'] = df['NumberofLanes'] * df['RoadType_ord']
print('Group 4 done.')

In [ ]:
# ══ GROUP 5 — WEATHER & ENVIRONMENT ══
# Impute Temperature: geohash median -> global median (train-fit)
geo_temp = train.groupby('geohash')['Temperature'].median()
global_temp = train['Temperature'].median()
for df in [tr, te]:
    df['Temperature'] = df['Temperature'].fillna(df['geohash'].map(geo_temp)).fillna(global_temp)

# 29) Weather ordinal by avg demand impact (data-driven, ascending)
weather_order = tr.groupby('Weather')[TARGET].mean().sort_values().index.tolist()
weather_map = {w: i for i, w in enumerate(weather_order)}
print('Weather ordinal (by ascending mean demand):', weather_map)

# 31) weather severity score (physical severity: clearer -> lower)
severity_map = {'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3}

# 30) temp quantile bins (5 buckets) — fit edges on train, apply to test
tr['temp_binned'], temp_bins = pd.qcut(tr['Temperature'], 5, labels=False,
                                        retbins=True, duplicates='drop')
tr['temp_binned'] = tr['temp_binned'].astype(int)
te['temp_binned'] = pd.cut(te['Temperature'], bins=temp_bins, labels=False,
                            include_lowest=True)
te['temp_binned'] = te['temp_binned'].fillna(-1).astype(int)  # outside train range

for df in [tr, te]:
    df['Weather_ord']            = df['Weather'].map(weather_map).astype(int)
    df['weather_severity_score'] = df['Weather'].map(severity_map).astype(int)
    # 32) weather x peak-hour, 33) temp x weather interactions
    df['weather_x_peak_hour'] = df['weather_severity_score'] * df['is_peak_hour']
    df['temp_x_weather']      = df['Temperature'] * df['weather_severity_score']
print('Group 5 done.')

In [ ]:
# ══ GROUP 3 — TARGET ENCODING (5-fold out-of-fold, leak-safe) ══
def oof_target_encode(group_cols, agg, new_col):
    """OOF encode train (per fold, excluding own fold); test uses full-train stats.
    Unseen groups fall back to the global statistic."""
    g = tr[TARGET]
    global_val = {'mean': g.mean(), 'median': g.median(),
                  'std': g.std(), 'max': g.max(), 'min': g.min()}[agg]
    # --- train OOF ---
    tr[new_col] = np.nan
    for f in range(FE_FOLDS):
        bank = tr[tr['_fold'] != f]
        stat = (bank.groupby(group_cols, observed=True)[TARGET]
                    .agg(agg).rename(new_col).reset_index())
        held = tr[tr['_fold'] == f][group_cols].reset_index(drop=True)
        vals = held.merge(stat, on=group_cols, how='left')[new_col].values
        tr.loc[tr['_fold'] == f, new_col] = vals
    tr[new_col] = tr[new_col].fillna(global_val)
    # --- test: full-train stats ---
    stat_full = (tr.groupby(group_cols, observed=True)[TARGET]
                   .agg(agg).rename(new_col).reset_index())
    te[new_col] = (te[group_cols].merge(stat_full, on=group_cols, how='left')[new_col]
                     .fillna(global_val).values)

te_specs = [
    (['geohash'],                'mean',   'te_geohash'),                  # 14
    (['geohash', 'hour'],        'mean',   'te_geohash_hour'),             # 15
    (['geohash', 'day'],         'mean',   'te_geohash_day'),              # 16
    (['geohash', 'time_of_day'], 'mean',   'te_geohash_timeofday'),        # 17
    (['RoadType', 'hour'],       'mean',   'te_roadtype_hour'),            # 18
    (['Weather', 'hour'],        'mean',   'te_weather_hour'),             # 19
    (['geohash4'],               'mean',   'te_geohash4'),                 # 20
    (['geohash', 'hour'],        'median', 'te_median_geohash_hour'),      # 21
    (['geohash', 'hour'],        'std',    'te_std_geohash_hour'),         # 22
    (['geohash'],                'max',    'te_max_geohash'),              # 23
]
for cols, agg, name in te_specs:
    oof_target_encode(cols, agg, name)
    print(f'  {name:<26} <- {agg:<6} of demand by {cols}')
print('Group 3 done (10 OOF target-encoded features).')

In [ ]:
# ══ GROUP 6 — NEIGHBOR & CONTEXT ══
# 34) 8 neighbors per geohash via cell-center perturbation (robust across pygeohash versions)
def get_neighbors(gh):
    d = pgh.decode_exactly(gh)
    p = len(gh)
    out = []
    for dla in (-1, 0, 1):
        for dlo in (-1, 0, 1):
            if dla == 0 and dlo == 0:
                continue
            nlat = d.latitude  + dla * 2 * d.latitude_error
            nlon = d.longitude + dlo * 2 * d.longitude_error
            out.append(pgh.encode(nlat, nlon, precision=p))
    return out

geo_mean_full = train.groupby('geohash')[TARGET].mean()  # full-train, train-only target
gm = geo_mean_full.to_dict()
global_mean = train[TARGET].mean()

neigh_mean, neigh_max = {}, {}
for gh in all_geo:
    vals = [gm[n] for n in get_neighbors(gh) if n in gm]
    if vals:
        neigh_mean[gh], neigh_max[gh] = float(np.mean(vals)), float(np.max(vals))
    else:
        neigh_mean[gh], neigh_max[gh] = global_mean, global_mean

for df in [tr, te]:
    df['mean_neighbor_demand'] = df['geohash'].map(neigh_mean)          # 35
    df['max_neighbor_demand']  = df['geohash'].map(neigh_max)           # 36
    own = df['geohash'].map(gm).fillna(global_mean)
    df['neighbor_demand_ratio'] = own / (df['mean_neighbor_demand'] + 1e-6)  # 37
print('Group 6 done.')

In [ ]:
# ══ GROUP 7 — FREQUENCY ENCODING (counts from train; unseen -> 0) ══
# 38) geohash frequency
geo_freq = train['geohash'].value_counts()
# 39) geohash x hour frequency (train, using parsed hour)
_tr_hour = train['timestamp'].str.split(':').str[0].astype(int)
gh_hour_freq = train.assign(_h=_tr_hour).groupby(['geohash', '_h']).size()
gh_hour_freq = gh_hour_freq.rename('geohash_hour_frequency').reset_index()
gh_hour_freq.columns = ['geohash', 'hour', 'geohash_hour_frequency']
# 40) weather / roadtype frequency (use imputed values from tr to stay consistent)
weather_freq  = tr.loc[tr['_fold'] >= -1, 'Weather'].value_counts()  # all train rows
roadtype_freq = tr['RoadType'].value_counts()

for df in [tr, te]:
    df['geohash_frequency']  = df['geohash'].map(geo_freq).fillna(0).astype(int)
    df['weather_frequency']  = df['Weather'].map(weather_freq).fillna(0).astype(int)
    df['roadtype_frequency'] = df['RoadType'].map(roadtype_freq).fillna(0).astype(int)

tr = tr.merge(gh_hour_freq, on=['geohash', 'hour'], how='left')
te = te.merge(gh_hour_freq, on=['geohash', 'hour'], how='left')
tr['geohash_hour_frequency'] = tr['geohash_hour_frequency'].fillna(0).astype(int)
te['geohash_hour_frequency'] = te['geohash_hour_frequency'].fillna(0).astype(int)
print('Group 7 done.')

In [ ]:
# ══ ASSEMBLY — build X / X_test, drop zero-variance, save CSVs, importance preview ══
FEATURES = [
    # Group 1 — temporal
    'hour', 'minute', 'time_slot', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'is_peak_hour', 'is_night', 'time_of_day', 'day_of_week', 'day_sin', 'day_cos', 'is_weekend',
    # Group 2 — geospatial
    'lat', 'lon', 'dist_from_center', 'geohash5_enc', 'geohash4_enc', 'geo_cluster',
    # Group 3 — target encoding (OOF)
    'te_geohash', 'te_geohash_hour', 'te_geohash_day', 'te_geohash_timeofday',
    'te_roadtype_hour', 'te_weather_hour', 'te_geohash4',
    'te_median_geohash_hour', 'te_std_geohash_hour', 'te_max_geohash',
    # Group 4 — road / infra
    'NumberofLanes', 'LargeVehicles_bin', 'Landmarks_bin',
    'road_capacity_score', 'infrastructure_score', 'RoadType_ord', 'lanes_x_roadtype',
    # Group 5 — weather / env
    'Weather_ord', 'Temperature', 'temp_binned', 'weather_severity_score',
    'weather_x_peak_hour', 'temp_x_weather',
    # Group 6 — neighbor / context
    'mean_neighbor_demand', 'max_neighbor_demand', 'neighbor_demand_ratio',
    # Group 7 — frequency
    'geohash_frequency', 'geohash_hour_frequency', 'weather_frequency', 'roadtype_frequency',
]

X      = tr[FEATURES].copy()
y      = tr[TARGET].values
X_test = te[FEATURES].copy()

# residual NaN safety (train medians)
med = X.median()
X      = X.fillna(med)
X_test = X_test.fillna(med)

# drop zero-variance columns (e.g. day-derived cols constant because day is fixed per split)
zero_var = [c for c in FEATURES if X[c].nunique() <= 1]
if zero_var:
    print('Dropping zero-variance columns:', zero_var)
    FEATURES = [c for c in FEATURES if c not in zero_var]
    X, X_test = X[FEATURES], X_test[FEATURES]

print(f'\nFinal feature count : {len(FEATURES)}')
print(f'X shape             : {X.shape}')
print(f'X_test shape        : {X_test.shape}')
print(f'NaN in X / X_test   : {X.isnull().sum().sum()} / {X_test.isnull().sum().sum()}')

# save feature sets
train_out = X.copy(); train_out['demand'] = y; train_out.insert(0, 'Index', tr['Index'].values)
test_out  = X_test.copy(); test_out.insert(0, 'Index', te['Index'].values)
train_out.to_csv(f'{OUT}/features/train_features.csv', index=False)
test_out.to_csv(f'{OUT}/features/test_features.csv', index=False)
print(f'Saved -> {OUT}/features/train_features.csv  {train_out.shape}')
print(f'Saved -> {OUT}/features/test_features.csv   {test_out.shape}')

In [ ]:
# Feature importance preview — quick LightGBM on the assembled features
_prev = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63,
                          random_state=SEED, n_jobs=-1, verbose=-1)
_prev.fit(X, y)
prev_imp = (pd.DataFrame({'feature': FEATURES, 'importance': _prev.feature_importances_})
              .sort_values('importance', ascending=False).reset_index(drop=True))

from sklearn.metrics import r2_score as _r2
print(f'Quick in-sample R² (sanity, not CV): {_r2(y, _prev.predict(X)):.5f}\n')
print('Top 20 features by importance:')
print(prev_imp.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=prev_imp.head(20), x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Phase 4 Feature Importance Preview (top 20)')
plt.tight_layout()
plt.savefig(f'{OUT}/features/phase4_importance_preview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 5 — Modeling Setup & Validation Strategy

Loads the engineered features from `/features/`, defines the metric and CV.

**Validation:** aligned `KFold(5, shuffle=True, random_state=42)` — *identical* to the Phase 4 target-encoding folds, so OOF target encoding and model CV use the same splits (leak-free). Test is ~99% seen geohashes, so this mirrors the leaderboard.

**Metric:** `R²` (sklearn). **Competition score:** `max(0, 100 · R²)`.

**Checkpointing (so a stopped run resumes, not restarts):** Optuna studies persist to `models/optuna_{lgb,xgb,cat}.db` (completed trials survive); each finished CV fold is saved to `models/checkpoints/cv_*.pkl`. Re-running a cell tops trials/folds back up to target. Set `FRESH_START=True` in cell 5.0 to wipe them and retune from scratch.

In [ ]:
# 5.0 — Load engineered features + set up CV / metric / checkpointing
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from scipy.optimize import minimize
from optuna.trial import TrialState
from optuna.samplers import TPESampler
import optuna, joblib, time, glob
optuna.logging.set_verbosity(optuna.logging.WARNING)

CKPT_DIR = f'{OUT}/models/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Set True to wipe checkpoints/studies and retune from scratch; False resumes where you left off.
FRESH_START = False
if FRESH_START:
    for f in glob.glob(f'{CKPT_DIR}/*') + glob.glob(f'{OUT}/models/optuna_*.db'):
        os.remove(f)
    print('FRESH_START: cleared checkpoints and Optuna studies.')

train_feat = pd.read_csv(f'{OUT}/features/train_features.csv')
test_feat  = pd.read_csv(f'{OUT}/features/test_features.csv')

FEATURES   = [c for c in train_feat.columns if c not in ('Index', 'demand')]
X          = train_feat[FEATURES].copy()
y          = train_feat['demand'].values
X_test     = test_feat[FEATURES].copy()
test_index = test_feat['Index'].values

CAT_FEATURES = [c for c in ['geo_cluster','geohash5_enc','geohash4_enc',
                            'time_of_day','day_of_week','temp_binned'] if c in FEATURES]
for c in CAT_FEATURES:
    X[c]      = X[c].astype(int)
    X_test[c] = X_test[c].astype(int)

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)   # == Phase 4 fe_kf

def comp_score(actual, pred):
    return max(0.0, 100.0 * r2_score(actual, pred))
def rmse(a, p):
    return np.sqrt(mean_squared_error(a, p))

_tr_i, _va_i = next(kf.split(X))
Xtt, Xtv = X.iloc[_tr_i], X.iloc[_va_i]
ytt, ytv = y[_tr_i], y[_va_i]

def make_cb(name, every=25):
    def _cb(study, trial):
        if (trial.number + 1) % every == 0:
            print(f'    [{name}] trial {trial.number+1:3d}  best R²={study.best_value:.5f}')
    return _cb

print(f'X={X.shape}  X_test={X_test.shape}  features={len(FEATURES)}')
print(f'CatBoost cat_features: {CAT_FEATURES}')

# ── Resumable Optuna study: trials persist in SQLite; reruns top up to n_target ──
def run_study(name, objective, n_target):
    storage = f'sqlite:///{OUT}/models/optuna_{name}.db'
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                study_name=name, storage=storage, load_if_exists=True)
    done = sum(t.state == TrialState.COMPLETE for t in study.trials)
    remaining = max(0, n_target - done)
    print(f'    [{name}] {done}/{n_target} trials complete; running {remaining} more')
    if remaining:
        study.optimize(objective, n_trials=remaining, callbacks=[make_cb(name)],
                       show_progress_bar=False)
    return study

# ── Checkpointed 5-fold OOF: each finished fold is saved; reruns skip done folds ──
def run_cv(make_model, fit_one, name, save=True):
    ckpt = f'{CKPT_DIR}/cv_{name}.pkl'
    if os.path.exists(ckpt):
        st = joblib.load(ckpt)
        oof, test_pred = st['oof'], st['test_pred']
        scores, done, models = st['scores'], st['done'], st['models']
        print(f'    [{name}] resume — folds done: {sorted(done)}')
    else:
        oof = np.zeros(len(X)); test_pred = np.zeros(len(X_test))
        scores, done, models = {}, set(), []
    for fold, (ti, vi) in enumerate(kf.split(X), 1):
        if fold in done:
            continue
        m = make_model()
        fit_one(m, X.iloc[ti], y[ti], X.iloc[vi], y[vi])
        oof[vi]    = m.predict(X.iloc[vi])
        test_pred += m.predict(X_test) / N_FOLDS
        scores[fold] = r2_score(y[vi], oof[vi]); done.add(fold); models.append(m)
        print(f'    [{name}] fold {fold}: R²={scores[fold]:.5f}  RMSE={rmse(y[vi], oof[vi]):.5f}')
        joblib.dump({'oof': oof, 'test_pred': test_pred, 'scores': scores,
                     'done': done, 'models': models}, ckpt)   # <-- checkpoint each fold
    sc = [scores[f] for f in sorted(scores)]
    print(f'    [{name}] OOF R²={r2_score(y, oof):.5f} | per-fold {np.mean(sc):.5f} ± {np.std(sc):.5f}')
    if save:
        joblib.dump(models, f'{OUT}/models/{name}_fold_models.pkl')
    return oof, test_pred, sc, models

---
## Phase 6 — Model 1: LightGBM (Optuna 150 trials)

In [ ]:
# 6.1 — LightGBM Optuna tuning (resumable; single holdout, early stopping)
def lgb_objective(trial):
    p = dict(objective='regression', metric='rmse',
             num_leaves        = trial.suggest_int('num_leaves', 31, 512),
             learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
             feature_fraction  = trial.suggest_float('feature_fraction', 0.5, 1.0),
             min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
             bagging_fraction=0.8, bagging_freq=5, n_estimators=1500,
             random_state=SEED, n_jobs=-1, verbose=-1)
    m = lgb.LGBMRegressor(**p)
    m.fit(Xtt, ytt, eval_set=[(Xtv, ytv)],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return r2_score(ytv, m.predict(Xtv))

t0 = time.time()
study_lgb = run_study('lgb', lgb_objective, 150)
print(f'\nLGB best holdout R²={study_lgb.best_value:.5f}  ({time.time()-t0:.0f}s)')
print('LGB best params:', study_lgb.best_params)

best_lgb = dict(study_lgb.best_params)
best_lgb.update(objective='regression', metric='rmse', bagging_fraction=0.8,
                bagging_freq=5, n_estimators=3000, random_state=SEED, n_jobs=-1, verbose=-1)
json.dump(best_lgb, open(f'{OUT}/models/best_lgb_params.json', 'w'), indent=2)

In [ ]:
# 6.2 — LightGBM 5-fold OOF
def fit_lgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(100, verbose=False)])

print('LightGBM 5-fold:')
oof_lgb, test_lgb, sc_lgb, models_lgb = run_cv(
    lambda: lgb.LGBMRegressor(**best_lgb), fit_lgb, 'lgb')

---
## Phase 7 — Model 2: XGBoost (Optuna 100 trials)

In [ ]:
# 7.1 — XGBoost Optuna tuning (resumable)
def xgb_objective(trial):
    p = dict(n_estimators=1500,
             learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
             max_depth        = trial.suggest_int('max_depth', 3, 12),
             subsample        = trial.suggest_float('subsample', 0.5, 1.0),
             colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
             tree_method='hist', random_state=SEED, n_jobs=-1,
             early_stopping_rounds=100, eval_metric='rmse')
    m = xgb.XGBRegressor(**p)
    m.fit(Xtt, ytt, eval_set=[(Xtv, ytv)], verbose=False)
    return r2_score(ytv, m.predict(Xtv))

t0 = time.time()
study_xgb = run_study('xgb', xgb_objective, 100)
print(f'\nXGB best holdout R²={study_xgb.best_value:.5f}  ({time.time()-t0:.0f}s)')
print('XGB best params:', study_xgb.best_params)

best_xgb = dict(study_xgb.best_params)
best_xgb.update(n_estimators=3000, tree_method='hist', random_state=SEED, n_jobs=-1,
                early_stopping_rounds=100, eval_metric='rmse')
json.dump(best_xgb, open(f'{OUT}/models/best_xgb_params.json', 'w'), indent=2)

In [ ]:
# 7.2 — XGBoost 5-fold OOF
def fit_xgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)

print('XGBoost 5-fold:')
oof_xgb, test_xgb, sc_xgb, models_xgb = run_cv(
    lambda: xgb.XGBRegressor(**best_xgb), fit_xgb, 'xgb')

---
## Phase 8 — Model 3: CatBoost (Optuna 100 trials)

In [ ]:
# 8.1 — CatBoost Optuna tuning (resumable; with cat_features)
def cat_objective(trial):
    p = dict(iterations=1000, learning_rate=0.05,
             depth               = trial.suggest_int('depth', 4, 10),
             l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1, 10),
             bagging_temperature = trial.suggest_float('bagging_temperature', 0, 1),
             random_seed=SEED, eval_metric='RMSE', od_type='Iter', od_wait=100, verbose=False)
    m = CatBoostRegressor(**p)
    m.fit(Xtt, ytt, eval_set=(Xtv, ytv), cat_features=CAT_FEATURES,
          use_best_model=True, verbose=False)
    return r2_score(ytv, m.predict(Xtv))

t0 = time.time()
study_cat = run_study('cat', cat_objective, 100)
print(f'\nCAT best holdout R²={study_cat.best_value:.5f}  ({time.time()-t0:.0f}s)')
print('CAT best params:', study_cat.best_params)

best_cat = dict(study_cat.best_params)
best_cat.update(iterations=3000, learning_rate=0.05, random_seed=SEED,
                eval_metric='RMSE', od_type='Iter', od_wait=100, verbose=False)
json.dump(best_cat, open(f'{OUT}/models/best_cat_params.json', 'w'), indent=2)

In [ ]:
# 8.2 — CatBoost 5-fold OOF
def fit_cat(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=CAT_FEATURES,
          use_best_model=True, verbose=False)

print('CatBoost 5-fold:')
oof_cat, test_cat, sc_cat, models_cat = run_cv(
    lambda: CatBoostRegressor(**best_cat), fit_cat, 'cat')

---
## Phase 9 — Stacking Ensemble
Ridge meta-learner on OOF predictions (honest meta-CV via `cross_val_predict`) **and** a scipy-optimized weighted blend. The higher-OOF-R² option is used for the final test prediction.

In [ ]:
# 9.1 — Stack OOF predictions; Ridge meta-learner + weighted blend
S_oof  = np.column_stack([oof_lgb, oof_xgb, oof_cat])
S_test = np.column_stack([test_lgb, test_xgb, test_cat])

# (a) Ridge meta-learner — honest OOF via cross_val_predict on the meta-features
ridge = Ridge(alpha=1.0)
meta_oof  = cross_val_predict(ridge, S_oof, y, cv=kf)
ridge.fit(S_oof, y)
meta_test = ridge.predict(S_test)
r2_meta   = r2_score(y, meta_oof)
print(f'Ridge meta  : coef={ridge.coef_.round(4)} intercept={ridge.intercept_:.5f}')
print(f'Ridge stack OOF R² = {r2_meta:.5f}')

# (b) Weighted blend — maximize R² over non-negative weights summing to 1
def neg_r2(w):
    w = np.clip(w, 0, None); s = w.sum()
    w = w / s if s > 0 else w
    return -r2_score(y, S_oof @ w)
res = minimize(neg_r2, [1/3, 1/3, 1/3], method='Nelder-Mead',
               options={'xatol': 1e-7, 'fatol': 1e-9, 'maxiter': 3000})
w = np.clip(res.x, 0, None); w = w / w.sum()
blend_oof  = S_oof  @ w
blend_test = S_test @ w
r2_blend   = r2_score(y, blend_oof)
print(f'Blend weights: LGB={w[0]:.3f} XGB={w[1]:.3f} CAT={w[2]:.3f}')
print(f'Weighted blend OOF R² = {r2_blend:.5f}')

# (c) choose the better strategy for the final prediction
if r2_meta >= r2_blend:
    final_oof, final_test, chosen = meta_oof, meta_test, 'Ridge stack'
else:
    final_oof, final_test, chosen = blend_oof, blend_test, f'Blend (LGB={w[0]:.2f},XGB={w[1]:.2f},CAT={w[2]:.2f})'
final_test = np.clip(final_test, 0, 1)

print(f'\n>>> Chosen ensemble : {chosen}')
print(f'>>> Final ensemble OOF R²       = {r2_score(y, final_oof):.5f}')
print(f'>>> Final competition score      = {comp_score(y, final_oof):.4f}  (max 0, 100·R²)')

# save stacking artifacts
joblib.dump({'ridge': ridge, 'weights': w, 'chosen': chosen,
             'r2_meta': r2_meta, 'r2_blend': r2_blend},
            f'{OUT}/models/stacking.pkl')
np.save(f'{OUT}/models/oof_preds.npy', S_oof)
np.save(f'{OUT}/models/test_preds.npy', S_test)
print('Saved stacking.pkl, oof_preds.npy, test_preds.npy')

---
## Phase 10 — SHAP Feature Importance (per model) & Comparison

In [ ]:
# 10.1 — Top-20 SHAP importance per model (fold-0 models; sampled rows)
import shap
from catboost import Pool

samp = X.sample(min(3000, len(X)), random_state=SEED)
shap_tables = {}

def top_shap(name, model, kind):
    try:
        if kind == 'cat':
            pool = Pool(samp, cat_features=CAT_FEATURES)
            sv = model.get_feature_importance(pool, type='ShapValues')[:, :-1]
        else:
            sv = shap.TreeExplainer(model).shap_values(samp)
        imp = np.abs(sv).mean(axis=0)
        src = 'SHAP'
    except Exception as e:
        print(f'  [{name}] SHAP failed ({type(e).__name__}); using native importance')
        imp = model.feature_importances_; src = 'native'
    tbl = (pd.DataFrame({'feature': FEATURES, 'importance': imp})
             .sort_values('importance', ascending=False).reset_index(drop=True))
    shap_tables[name] = (tbl, src)
    return tbl

tbl_lgb = top_shap('LightGBM', models_lgb[0], 'lgb')
tbl_xgb = top_shap('XGBoost',  models_xgb[0], 'xgb')
tbl_cat = top_shap('CatBoost', models_cat[0], 'cat')

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for ax, (name, (tbl, src)) in zip(axes, shap_tables.items()):
    sns.barplot(data=tbl.head(20), x='importance', y='feature', palette='viridis', ax=ax)
    ax.set_title(f'{name} — Top 20 ({src})')
    ax.set_xlabel(f'mean(|{src}|)')
plt.tight_layout()
plt.savefig(f'{OUT}/features/shap_per_model.png', dpi=150, bbox_inches='tight')
plt.show()

for name, (tbl, src) in shap_tables.items():
    print(f'\n=== {name} top 20 ({src}) ===')
    print(tbl.head(20).to_string(index=False))

In [ ]:
# 10.2 — Model comparison (CV R² mean ± std)
rows = [
    ('LightGBM',       r2_score(y, oof_lgb), np.mean(sc_lgb), np.std(sc_lgb)),
    ('XGBoost',        r2_score(y, oof_xgb), np.mean(sc_xgb), np.std(sc_xgb)),
    ('CatBoost',       r2_score(y, oof_cat), np.mean(sc_cat), np.std(sc_cat)),
    ('Ridge stack',    r2_meta, np.nan, np.nan),
    ('Weighted blend', r2_blend, np.nan, np.nan),
]
comp = pd.DataFrame(rows, columns=['model', 'oof_r2', 'fold_mean', 'fold_std'])
comp['comp_score'] = (100 * comp['oof_r2']).clip(lower=0)
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#4CAF50' if v == comp['oof_r2'].max() else '#2196F3' for v in comp['oof_r2']]
ax.barh(comp['model'], comp['oof_r2'], color=colors, edgecolor='white')
ax.set_xlim(min(comp['oof_r2']) - 0.005, 1.0)
ax.axvline(0.95, color='red', linestyle='--', label='target R²=0.95')
for i, v in enumerate(comp['oof_r2']):
    ax.text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)
ax.set_title('OOF R² by model'); ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Phase 11 — Generate Submission

In [ ]:
# 11.1 — Build submission from the chosen ensemble
submission = pd.DataFrame({'Index': test_index, 'demand': final_test})
sub_path = f'{OUT}/submissions/submission_stack.csv'
submission.to_csv(sub_path, index=False)

print(f'Saved -> {sub_path}  shape={submission.shape}')
print(f'pred: min={final_test.min():.5f} max={final_test.max():.5f} '
      f'mean={final_test.mean():.5f} median={np.median(final_test):.5f}')
submission.head(10)

In [ ]:
# 11.2 — Prediction distribution + final summary
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(y, bins=80, alpha=0.6, label='train actual', color='steelblue')
axes[0].hist(final_test, bins=80, alpha=0.6, label='test pred', color='coral')
axes[0].set_title('Demand: train vs predicted'); axes[0].legend()
axes[1].hist(np.log1p(y), bins=80, alpha=0.6, label='train actual', color='steelblue')
axes[1].hist(np.log1p(final_test), bins=80, alpha=0.6, label='test pred', color='coral')
axes[1].set_title('log1p(demand): train vs predicted'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{OUT}/submissions/prediction_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('=' * 64)
print('FINAL RESULTS')
print('=' * 64)
print(f'  LightGBM       OOF R² = {r2_score(y, oof_lgb):.5f}')
print(f'  XGBoost        OOF R² = {r2_score(y, oof_xgb):.5f}')
print(f'  CatBoost       OOF R² = {r2_score(y, oof_cat):.5f}')
print(f'  Ridge stack    OOF R² = {r2_meta:.5f}')
print(f'  Weighted blend OOF R² = {r2_blend:.5f}')
print('-' * 64)
print(f'  CHOSEN         : {chosen}')
print(f'  Final OOF R²   : {r2_score(y, final_oof):.5f}')
print(f'  Competition    : {comp_score(y, final_oof):.4f}  (target > 95)')
print('=' * 64)
print('Models saved to /models/:')
import os as _os
for f in sorted(_os.listdir(f'{OUT}/models')):
    print('   ', f)

---
## Phase 12 — Error Analysis
Diagnose OOF residuals across time, space, and feature segments to drive targeted improvements.

In [ ]:
# 12.0 — Load OOF predictions + raw train for error analysis
from scipy.stats import skew as _skew, kurtosis as _kurt, probplot

train_raw = pd.read_csv(f'{BASE}/train.csv')
oof_stack = np.load(f'{OUT}/models/oof_preds.npy')       # (N, 3) LGB/XGB/CAT
stk       = joblib.load(f'{OUT}/models/stacking.pkl')
oof_final = np.clip(stk['ridge'].predict(oof_stack), 0, 1)

edf = train_raw.copy()
edf['pred']     = oof_final
edf['err']      = oof_final - edf['demand']
edf['abs_err']  = edf['err'].abs()
edf['sq_err']   = edf['err'] ** 2
parts           = edf['timestamp'].str.split(':')
edf['hour']     = parts.str[0].astype(int)
edf['minute']   = parts.str[1].astype(int)
edf['time_slot']= edf['hour']*4 + edf['minute']//15
edf['geo_cluster'] = train_feat['geo_cluster'].values

print(f"OOF R²  : {r2_score(y, oof_final):.5f}")
print(f"RMSE    : {np.sqrt(edf['sq_err'].mean()):.5f}")
print(f"Residual skew={_skew(edf['err']):.4f}  kurt={_kurt(edf['err']):.2f}")
print(f"Mean signed error: {edf['err'].mean():.5f}  (>0=overpredict)")

In [ ]:
# 12.1 — Actual vs Predicted + Residual distribution
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# scatter (sample 6000 for clarity)
samp = edf.sample(6000, random_state=SEED)
sc = axes[0].scatter(samp['demand'], samp['pred'],
                     c=samp['abs_err'], cmap='YlOrRd', s=8, alpha=0.5)
plt.colorbar(sc, ax=axes[0], label='|error|')
axes[0].plot([0,1],[0,1],'b--',lw=1.5,label='perfect')
axes[0].set_title(f'Actual vs Predicted  R²={r2_score(y, oof_final):.4f}')
axes[0].set_xlabel('Actual demand'); axes[0].set_ylabel('Predicted')
axes[0].legend(fontsize=8)

# residual histogram
axes[1].hist(edf['err'], bins=120, color='steelblue', edgecolor='none')
axes[1].axvline(0, color='red', lw=1.5)
axes[1].set_title(f'Residual dist  skew={_skew(edf["err"]):.3f}  kurt={_kurt(edf["err"]):.1f}')
axes[1].set_xlabel('prediction error')

# Q-Q plot
res_sorted = np.sort(edf['err'])
n = len(res_sorted)
q_norm = np.linspace(0.001, 0.999, n)
from scipy.stats import norm
axes[2].scatter(norm.ppf(q_norm), res_sorted, s=1, alpha=0.3, color='coral')
axes[2].plot([-4,4], [np.percentile(edf['err'],0.1*100),
              np.percentile(edf['err'],99.9)], 'b--', lw=1.5)
axes[2].set_title('Q-Q plot of residuals (heavy tails = large kurt)')
axes[2].set_xlabel('Normal quantile'); axes[2].set_ylabel('Residual')

plt.tight_layout()
plt.savefig(f'{OUT}/notebooks/E12_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12.2 — Error by hour (RMSE + signed mean)
hr = edf.groupby('hour').apply(lambda g: pd.Series({
    'rmse': np.sqrt((g['sq_err']).mean()),
    'mean_err': g['err'].mean(),
    'n': len(g)
})).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 4))
axes[0].bar(hr['hour'], hr['rmse'], color='steelblue')
axes[0].set_title('RMSE by Hour'); axes[0].set_xlabel('Hour')
axes[0].set_xticks(range(24))

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in hr['mean_err']]
axes[1].bar(hr['hour'], hr['mean_err'], color=colors)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Mean Signed Error by Hour (red=overpredict, green=underpredict)')
axes[1].set_xlabel('Hour'); axes[1].set_xticks(range(24))

plt.tight_layout()
plt.savefig(f'{OUT}/notebooks/E12_error_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Worst 5 hours by RMSE:')
print(hr.sort_values('rmse',ascending=False).head(5).to_string(index=False))

In [ ]:
# 12.3 — Error by geo_cluster + RoadType
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

gc = edf.groupby('geo_cluster').apply(lambda g: pd.Series({
    'rmse': np.sqrt((g['sq_err']).mean()),
    'mean_err': g['err'].mean(), 'n': len(g)
})).reset_index().sort_values('rmse', ascending=True)

colors_gc = ['#e74c3c' if r > gc['rmse'].quantile(0.8) else '#3498db' for r in gc['rmse']]
axes[0].barh(gc['geo_cluster'].astype(str), gc['rmse'], color=colors_gc)
axes[0].set_title('RMSE by Geo Cluster (red = worst 20%)')
axes[0].set_xlabel('RMSE')

rt = edf.groupby('RoadType').apply(lambda g: pd.Series({
    'rmse': np.sqrt((g['sq_err']).mean()),
    'mean_err': g['err'].mean(), 'n': len(g)
})).reset_index().sort_values('rmse', ascending=False)
axes[1].bar(rt['RoadType'], rt['rmse'],
            color=['#e74c3c','#f39c12','#2ecc71'][:len(rt)])
axes[1].set_title('RMSE by RoadType — Highway is the dominant error source')
for i,(v,n) in enumerate(zip(rt['rmse'],rt['n'])):
    axes[1].text(i, v+0.001, f'n={int(n)}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT}/notebooks/E12_error_by_cluster_roadtype.png', dpi=150, bbox_inches='tight')
plt.show()
print(rt.to_string(index=False))

In [ ]:
# 12.4 — Error by Weather type
wth = edf.groupby('Weather').apply(lambda g: pd.Series({
    'rmse': np.sqrt((g['sq_err']).mean()),
    'mean_err': g['err'].mean(), 'n': len(g)
})).reset_index().sort_values('rmse', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(wth['Weather'], wth['rmse'], color=['#f39c12','#3498db','#95a5a6','#1abc9c'])
ax.set_title('RMSE by Weather (differences are small — weather is NOT the main error driver)')
for i,(v,n) in enumerate(zip(wth['rmse'],wth['n'])):
    ax.text(i, v+0.0002, f'n={int(n)}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT}/notebooks/E12_error_by_weather.png', dpi=150, bbox_inches='tight')
plt.show()
print(wth.to_string(index=False))

In [ ]:
# 12.5 — Top 50 worst samples + systematic bias check
worst = edf.nlargest(50,'abs_err')[['geohash','hour','RoadType','Weather',
                                    'demand','pred','err','geo_cluster']]
print('=== TOP 50 WORST SAMPLES ===')
print('RoadType dist  :', worst['RoadType'].value_counts().to_dict())
print('Weather dist   :', worst['Weather'].value_counts().to_dict())
print('Hour dist      :', worst['hour'].value_counts().sort_index().to_dict())
print('Cluster dist   :', worst['geo_cluster'].value_counts().head(5).to_dict())
print(f'Mean actual    : {worst["demand"].mean():.4f}  (population mean: {y.mean():.4f})')
print(f'Mean predicted : {worst["pred"].mean():.4f}  → UNDERPREDICTION by {worst["demand"].mean()-worst["pred"].mean():.4f}')

print('\n=== SYSTEMATIC BIAS BY SEGMENT ===')
for col in ['RoadType','Weather']:
    seg = edf.groupby(col)['err'].agg(['mean','std']).round(5)
    print(f'\n{col}:'); print(seg)

print('\n=== INSIGHT SUMMARY ===')
print('1. Highway RMSE = 0.074 (3.3x Residential) — 45/50 worst are Highway')
print('2. Worst clusters: 5, 12, 0 — highway-dense zones')
print('3. Residual kurt=19.9: extreme errors cluster at high-demand Highway rows')
print('4. Weather differences < 0.002 RMSE — skip weather sub-models')
print('5. log1p(demand) skew: 3.73 → 2.97 — separate log-space model worth trying')

---
## Phase 13 — Targeted Improvements
Driven by Phase 12 findings:

| Finding | Strategy |
|---------|----------|
| Highway RMSE 3.3× worse; 45/50 worst samples are Highway | **A** — Highway×geohash×hour OOF TE; `is_highway` flag |
| Peak hours h=10,13,14 have highest RMSE | **A** — `peak_hour×geohash` OOF TE |
| Residual skew −0.84, kurt 19.9, target skew 3.73 | **C** — separate log1p-target LGB to blend |
| Lag/rolling features absent | **E** — geo slot-lag means, EWMA, demand rank in cluster |
| Clusters 0,5,12 have RMSE > 0.042 | **B** — per-geohash OOF bias correction |
| Weather effect negligible | skip D |


In [ ]:
# 13.1 — New features: lag slot means, EWMA, demand rank, highway flags
#         (fit on train only; applied to both train and test)

tr2 = train_feat.copy()
te2 = test_feat.copy()

# re-parse time cols on raw data (needed for lag/ewma calculations)
for df, raw in [(tr2, train_raw), (te2, pd.read_csv(f'{BASE}/test.csv'))]:
    parts = raw['timestamp'].str.split(':')
    df['_hour']     = parts.str[0].astype(int)
    df['_minute']   = parts.str[1].astype(int)
    df['_timeslot'] = df['_hour']*4 + df['_minute']//15
    df['_geohash']  = raw['geohash']
    df['_roadtype'] = raw['RoadType'].fillna(
        raw['geohash'].map(train_raw.groupby('geohash')['RoadType']
                          .agg(lambda s: s.mode().iat[0] if len(s.dropna()) else 'Residential'))
    ).fillna('Residential')

# --- E: lag-1, lag-2 slot demand (mean of geohash at prev time slots) ---
slot_geo_mean = (train_raw.assign(
    _ts=train_raw['timestamp'].str.split(':').str[0].astype(int)*4
      + train_raw['timestamp'].str.split(':').str[1].astype(int)//15
).groupby(['geohash','_ts'])['demand'].mean())

global_slot_mean = (train_raw.assign(
    _ts=train_raw['timestamp'].str.split(':').str[0].astype(int)*4
      + train_raw['timestamp'].str.split(':').str[1].astype(int)//15
).groupby('_ts')['demand'].mean())

def get_lag_demand(geohash_series, slot_series, lag):
    target_slot = slot_series - lag
    vals = []
    for gh, ts in zip(geohash_series, target_slot):
        ts_clipped = int(ts) % 96
        if (gh, ts_clipped) in slot_geo_mean:
            vals.append(slot_geo_mean[(gh, ts_clipped)])
        elif ts_clipped in global_slot_mean:
            vals.append(global_slot_mean[ts_clipped])
        else:
            vals.append(train_raw['demand'].mean())
    return np.array(vals)

for df in [tr2, te2]:
    df['lag1_slot_demand'] = get_lag_demand(df['_geohash'], df['_timeslot'], 1)
    df['lag2_slot_demand'] = get_lag_demand(df['_geohash'], df['_timeslot'], 2)
    df['lag4_slot_demand'] = get_lag_demand(df['_geohash'], df['_timeslot'], 4)  # 1-hr lag
print('Lag features done')

# --- E: EWMA per geohash over time slots (alpha=0.3) ---
alpha = 0.3
ewma_map = {}
for gh, grp in train_raw.assign(
    _ts=train_raw['timestamp'].str.split(':').str[0].astype(int)*4
      + train_raw['timestamp'].str.split(':').str[1].astype(int)//15
).groupby(['geohash','_ts'])['demand'].mean().groupby(level=0):
    vals = grp.sort_index()
    ewma_vals = vals.ewm(alpha=alpha).mean()
    ewma_map[gh] = dict(zip(vals.index.get_level_values('_ts'), ewma_vals))
global_ewma = train_raw['demand'].mean()

def get_ewma(geohash_series, slot_series):
    return np.array([
        ewma_map.get(gh, {}).get(int(ts) % 96, global_ewma)
        for gh, ts in zip(geohash_series, slot_series)
    ])

for df in [tr2, te2]:
    df['geo_ewma_demand'] = get_ewma(df['_geohash'], df['_timeslot'])
print('EWMA features done')

# --- E: demand rank of geohash within its geo_cluster ---
geo_mean_map  = train_raw.groupby('geohash')['demand'].mean().to_dict()
cluster_map_r = dict(zip(train_feat['_geohash'] if '_geohash' in train_feat.columns
                         else [train_raw.iloc[i]['geohash'] for i in range(len(train_raw))],
                         train_feat['geo_cluster']))
cluster_map_r = dict(zip(
    [train_raw.iloc[i]['geohash'] for i in range(len(train_raw))],
    train_feat['geo_cluster']
))
geo_cluster_ranks = {}
cluster_geo_means = pd.DataFrame({'geohash': list(geo_mean_map),
                                   'geo_mean': list(geo_mean_map.values())})
cluster_geo_means['geo_cluster'] = cluster_geo_means['geohash'].map(cluster_map_r)
cluster_geo_means['rank_in_cluster'] = (
    cluster_geo_means.groupby('geo_cluster')['geo_mean']
    .rank(method='average', pct=True)
)
rank_map = dict(zip(cluster_geo_means['geohash'], cluster_geo_means['rank_in_cluster']))
global_rank_med = cluster_geo_means['rank_in_cluster'].median()
for df in [tr2, te2]:
    df['demand_rank_in_cluster'] = df['_geohash'].map(rank_map).fillna(global_rank_med)
print('Demand rank in cluster done')

# --- A: Highway flag + Highway×geohash×hour OOF target encoding ---
for df in [tr2, te2]:
    df['is_highway'] = (df['_roadtype'] == 'Highway').astype(int)

# OOF encode: Highway × geohash × hour
tr2['_y'] = y
kf2 = KFold(n_splits=5, shuffle=True, random_state=SEED)
global_hw_mean = train_raw[train_raw['RoadType']=='Highway']['demand'].mean()

tr2['te_highway_geo_hour'] = np.nan
for f,(ti,vi) in enumerate(kf2.split(tr2),1):
    bank = tr2.iloc[ti]
    stat = (bank[bank['is_highway']==1]
            .groupby(['_geohash','_hour'])['_y'].mean()
            .reset_index().rename(columns={'_y':'te_highway_geo_hour'}))
    held = tr2.iloc[vi][['_geohash','_hour']].reset_index(drop=True)
    merged = held.merge(stat, on=['_geohash','_hour'], how='left')['te_highway_geo_hour'].values
    tr2.loc[tr2.index[vi],'te_highway_geo_hour'] = merged
tr2['te_highway_geo_hour'] = tr2['te_highway_geo_hour'].fillna(global_hw_mean)

stat_full = (tr2[tr2['is_highway']==1]
             .groupby(['_geohash','_hour'])['_y'].mean()
             .reset_index().rename(columns={'_y':'te_highway_geo_hour'}))
te2['te_highway_geo_hour'] = (
    te2[['_geohash','_hour']].merge(stat_full, on=['_geohash','_hour'], how='left')
    ['te_highway_geo_hour'].fillna(global_hw_mean).values
)
print('Highway×geo×hour TE done')

# --- A: peak_hour × geohash OOF TE ---
tr2['is_peak'] = tr2['is_peak_hour'] if 'is_peak_hour' in tr2.columns else (
    tr2['_hour'].isin([7,8,9,17,18,19,20]).astype(int))
te2['is_peak'] = te2['_hour'].isin([7,8,9,17,18,19,20]).astype(int)

tr2['te_peak_geohash'] = np.nan
for f,(ti,vi) in enumerate(kf2.split(tr2),1):
    bank = tr2.iloc[ti]
    stat = (bank[bank['is_peak']==1]
            .groupby('_geohash')['_y'].mean()
            .reset_index().rename(columns={'_y':'te_peak_geohash'}))
    held = tr2.iloc[vi][['_geohash']].reset_index(drop=True)
    merged = held.merge(stat, on='_geohash', how='left')['te_peak_geohash'].values
    tr2.loc[tr2.index[vi],'te_peak_geohash'] = merged
global_peak_mean = tr2[tr2['is_peak']==1]['_y'].mean()
tr2['te_peak_geohash'] = tr2['te_peak_geohash'].fillna(global_peak_mean)

stat_pk = (tr2[tr2['is_peak']==1].groupby('_geohash')['_y'].mean()
           .reset_index().rename(columns={'_y':'te_peak_geohash'}))
te2['te_peak_geohash'] = (
    te2[['_geohash']].merge(stat_pk, on='_geohash', how='left')
    ['te_peak_geohash'].fillna(global_peak_mean).values
)
print('Peak×geohash TE done')

# drop helper columns
for df in [tr2, te2]:
    df.drop(columns=[c for c in ['_hour','_minute','_timeslot','_geohash','_roadtype','_y','is_peak']
                     if c in df.columns], inplace=True, errors='ignore')

NEW_FEATURES = ['lag1_slot_demand','lag2_slot_demand','lag4_slot_demand',
                'geo_ewma_demand','demand_rank_in_cluster',
                'is_highway','te_highway_geo_hour','te_peak_geohash']
FEATURES_V2 = FEATURES + NEW_FEATURES
X2      = tr2[FEATURES_V2].fillna(tr2[FEATURES_V2].median())
X2_test = te2[FEATURES_V2].fillna(X2.median())

print(f'\nV2 feature count: {len(FEATURES_V2)}  (+{len(NEW_FEATURES)} new)')
print(f'X2={X2.shape}  X2_test={X2_test.shape}')
print('New features:', NEW_FEATURES)

In [ ]:
# 13.2 — Strategy C: LightGBM trained on log1p(demand) target
y_log = np.log1p(y)

# use same best_lgb params but train on log target
_tr_log, _va_log = next(KFold(5,shuffle=True,random_state=SEED).split(X2))
Xtt2, Xtv2 = X2.iloc[_tr_log], X2.iloc[_va_log]
ytt2, ytv2 = y_log[_tr_log], y_log[_va_log]

def lgb_log_objective(trial):
    p = dict(objective='regression', metric='rmse',
             num_leaves        = trial.suggest_int('num_leaves', 63, 512),
             learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
             feature_fraction  = trial.suggest_float('feature_fraction', 0.5, 1.0),
             min_child_samples = trial.suggest_int('min_child_samples', 5, 60),
             bagging_fraction=0.8, bagging_freq=5, n_estimators=1500,
             random_state=SEED, n_jobs=-1, verbose=-1)
    m = lgb.LGBMRegressor(**p)
    m.fit(Xtt2, ytt2, eval_set=[(Xtv2, ytv2)], callbacks=[lgb.early_stopping(100, verbose=False)])
    pred_orig = np.expm1(np.clip(m.predict(Xtv2), 0, None))
    return r2_score(np.expm1(ytv2), pred_orig)

study_log = run_study('lgb_log', lgb_log_objective, 60)
best_lgb_log = dict(study_log.best_params)
best_lgb_log.update(objective='regression', metric='rmse', bagging_fraction=0.8,
                    bagging_freq=5, n_estimators=3000, random_state=SEED, n_jobs=-1, verbose=-1)
print(f'log-LGB best holdout R²={study_log.best_value:.5f}')
print('params:', study_log.best_params)

In [ ]:
# 13.3 — Retrain all models on V2 features (5-fold OOF)
print('== LightGBM V2 ==')
oof_lgb2, test_lgb2, sc_lgb2, models_lgb2 = run_cv(
    lambda: lgb.LGBMRegressor(**best_lgb),
    lambda m,Xtr,ytr,Xva,yva: m.fit(
        Xtr, ytr, eval_set=[(Xva,yva)],
        callbacks=[lgb.early_stopping(100,verbose=False)]),
    'lgb_v2'
)

print('== XGBoost V2 ==')
oof_xgb2, test_xgb2, sc_xgb2, models_xgb2 = run_cv(
    lambda: xgb.XGBRegressor(**best_xgb),
    lambda m,Xtr,ytr,Xva,yva: m.fit(Xtr,ytr,eval_set=[(Xva,yva)],verbose=False),
    'xgb_v2'
)

print('== CatBoost V2 ==')
CAT_FEATURES_V2 = [c for c in CAT_FEATURES if c in FEATURES_V2]
for c in CAT_FEATURES_V2: X2[c]=X2[c].astype(int); X2_test[c]=X2_test[c].astype(int)
oof_cat2, test_cat2, sc_cat2, models_cat2 = run_cv(
    lambda: CatBoostRegressor(**best_cat),
    lambda m,Xtr,ytr,Xva,yva: m.fit(Xtr,ytr,eval_set=(Xva,yva),
                                    cat_features=CAT_FEATURES_V2,
                                    use_best_model=True,verbose=False),
    'cat_v2'
)

print('== Log-LGB V2 ==')
oof_log2 = np.zeros(len(X2)); test_log2 = np.zeros(len(X2_test)); sc_log2 = []
kf_log = KFold(5, shuffle=True, random_state=SEED)
for fold,(ti,vi) in enumerate(kf_log.split(X2),1):
    m = lgb.LGBMRegressor(**best_lgb_log)
    m.fit(X2.iloc[ti], y_log[ti], eval_set=[(X2.iloc[vi], y_log[vi])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_log2[vi] = np.expm1(np.clip(m.predict(X2.iloc[vi]), 0, None))
    test_log2   += np.expm1(np.clip(m.predict(X2_test), 0, None)) / 5
    s = r2_score(y[vi], oof_log2[vi]); sc_log2.append(s)
    print(f'    [log-lgb] fold {fold}: R²={s:.5f}')
print(f'    [log-lgb] OOF R²={r2_score(y, oof_log2):.5f} | {np.mean(sc_log2):.5f} ± {np.std(sc_log2):.5f}')

In [ ]:
# 13.4 — Strategy B: per-geohash OOF bias correction
#         Compute mean signed error per geohash from V2 ensemble OOF
#         Apply correction to test predictions ONLY for geohashes with enough OOF samples

# First build a preliminary V2 ensemble to get correctable OOF preds
S2_oof = np.column_stack([oof_lgb2, oof_xgb2, oof_cat2, oof_log2])
S2_test= np.column_stack([test_lgb2, test_xgb2, test_cat2, test_log2])

res2 = minimize(lambda w: -r2_score(y, np.column_stack([oof_lgb2,oof_xgb2,oof_cat2,oof_log2])
                @ np.clip(w,0,None) / max(np.clip(w,0,None).sum(),1e-9)),
               [0.25]*4, method='Nelder-Mead', options={'xatol':1e-7,'maxiter':5000})
w2 = np.clip(res2.x,0,None); w2 /= w2.sum()
oof_v2_prelim  = np.clip(S2_oof  @ w2, 0, 1)
test_v2_prelim = np.clip(S2_test @ w2, 0, 1)

# per-geohash bias from OOF (only apply to geohashes with >= 10 OOF samples)
geo_series = pd.Series(train_raw['geohash'].values)
geo_bias   = pd.DataFrame({'geohash': geo_series, 'err': oof_v2_prelim - y})
geo_bias_agg = geo_bias.groupby('geohash')['err'].agg(['mean','count']).reset_index()
geo_bias_agg.columns = ['geohash','geo_bias','n']
# only correct geohashes with >= 10 samples AND abs bias > 0.005
geo_bias_agg = geo_bias_agg[(geo_bias_agg['n'] >= 10) & (geo_bias_agg['geo_bias'].abs() > 0.005)]
bias_map = dict(zip(geo_bias_agg['geohash'], geo_bias_agg['geo_bias']))
print(f'Geohashes with correctable bias: {len(bias_map)}')

# apply correction: pred_corrected = pred - bias
test_geo   = pd.read_csv(f'{BASE}/test.csv')['geohash']
test_corr  = test_v2_prelim - test_geo.map(bias_map).fillna(0).values
test_corr  = np.clip(test_corr, 0, 1)

# OOF correction (for measuring impact)
oof_corr = oof_v2_prelim - geo_series.map(bias_map).fillna(0).values
oof_corr = np.clip(oof_corr, 0, 1)
print(f'Pre-correction  OOF R²: {r2_score(y, oof_v2_prelim):.5f}')
print(f'Post-correction OOF R²: {r2_score(y, oof_corr):.5f}')

In [ ]:
# 13.5 — Final V2 ensemble: Ridge + weighted blend, pick best
ridge2 = Ridge(alpha=1.0)
meta_oof2  = cross_val_predict(ridge2, S2_oof, y, cv=KFold(5,shuffle=True,random_state=SEED))
ridge2.fit(S2_oof, y)
meta_test2 = ridge2.predict(S2_test)
r2_meta2   = r2_score(y, meta_oof2)
print(f'Ridge V2 stack  OOF R² = {r2_meta2:.5f}')

r2_blend2  = r2_score(y, oof_v2_prelim)
print(f'Weighted blend  OOF R² = {r2_blend2:.5f}  weights={w2.round(3)}')

r2_corr2   = r2_score(y, oof_corr)
print(f'Bias-corrected blend R² = {r2_corr2:.5f}')

# pick best
options = {
    'Ridge V2':          (r2_meta2,  np.clip(meta_test2, 0, 1)),
    'Blend V2':          (r2_blend2, test_v2_prelim),
    'Bias-corrected V2': (r2_corr2,  test_corr),
}
best_name_v2 = max(options, key=lambda k: options[k][0])
best_r2_v2, best_test_v2 = options[best_name_v2]
print(f'\n>>> Best V2 option: {best_name_v2}  R²={best_r2_v2:.5f}')

joblib.dump({'ridge': ridge2, 'weights': w2, 'chosen': best_name_v2,
             'r2': best_r2_v2}, f'{OUT}/models/stacking_v2.pkl')
np.save(f'{OUT}/models/oof_preds_v2.npy', S2_oof)
np.save(f'{OUT}/models/test_preds_v2.npy', S2_test)

In [ ]:
# 13.6 — V1 vs V2 comparison
r2_v1 = stk['r2_meta']
rows = [
    ('V1 LGB',               r2_score(y, oof_lgb)),
    ('V1 XGB',               r2_score(y, oof_xgb)),
    ('V1 CAT',               r2_score(y, oof_cat)),
    ('V1 Ridge stack',        r2_v1),
    ('── V2 LGB',             r2_score(y, oof_lgb2)),
    ('── V2 XGB',             r2_score(y, oof_xgb2)),
    ('── V2 CAT',             r2_score(y, oof_cat2)),
    ('── V2 Log-LGB',         r2_score(y, oof_log2)),
    ('── V2 Ridge stack',     r2_meta2),
    ('── V2 Bias-corrected',  r2_corr2),
]
print('=' * 52)
print(f'  {"Model":<28}  {"OOF R²":>8}  {"Score":>8}')
print('=' * 52)
for name, r2 in rows:
    score = max(0, 100*r2)
    flag = ' *** BEST' if name.strip() == best_name_v2 else ''
    print(f'  {name:<28}  {r2:8.5f}  {score:8.4f}{flag}')
print('=' * 52)
print(f'  Improvement V1→V2: {best_r2_v2 - r2_v1:+.5f} R²  ({100*(best_r2_v2-r2_v1):+.4f} pts)')

fig, ax = plt.subplots(figsize=(12, 6))
names2 = [r[0] for r in rows]; scores2 = [r[1] for r in rows]
colors2 = ['#e74c3c' if 'V2' in n else '#95a5a6' for n in names2]
colors2 = ['#f1c40f' if n.strip() == best_name_v2 else c for n,c in zip(names2,colors2)]
ax.barh(names2, scores2, color=colors2, edgecolor='white')
ax.axvline(0.95, color='red', linestyle='--', label='target 0.95')
ax.axvline(r2_v1, color='grey', linestyle=':', label=f'V1 best {r2_v1:.4f}')
for i,v in enumerate(scores2): ax.text(v+0.0002, i, f'{v:.4f}', va='center', fontsize=8)
ax.set_title('V1 vs V2 OOF R² — yellow=best V2'); ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/notebooks/E13_v1_v2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 13.7 — Save improved submission
sub_v2 = pd.DataFrame({'Index': test_index, 'demand': best_test_v2})
sub_v2_path = f'{OUT}/submissions/submission_v2_improved.csv'
sub_v2.to_csv(sub_v2_path, index=False)

# compare pred distributions
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(np.load(f'{OUT}/models/test_preds.npy') @ stk['weights'],
        bins=80, alpha=0.5, label=f'V1 (score={100*r2_v1:.3f})', color='steelblue')
ax.hist(best_test_v2, bins=80, alpha=0.5,
        label=f'V2 {best_name_v2} (score={100*best_r2_v2:.3f})', color='coral')
ax.set_title('Test Prediction Distribution: V1 vs V2'); ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT}/submissions/v1_vs_v2_preds.png', dpi=150, bbox_inches='tight')
plt.show()

print('=' * 60)
print('IMPROVED SUBMISSION SAVED')
print(f'  Path  : {sub_v2_path}')
print(f'  Method: {best_name_v2}')
print(f'  OOF R²: {best_r2_v2:.5f}')
print(f'  Score : {100*best_r2_v2:.4f}  (V1 was {100*r2_v1:.4f})')
print(f'  Rows  : {len(sub_v2)}')
print('=' * 60)

---
## Phase 14 — Final Submission

Retrains all three models on the **entire training set** (n_estimators=3000) using the best hyperparameters from Optuna, then blends with the Ridge stacking weights learned during CV. This is standard practice: CV finds the right hyperparameters and blend weights, full-data training gives the strongest model (no rows held out).

| Step | Detail |
|------|---------|
| Features | V1 features from `/features/train_features.csv` (50 features) |
| Models | LGB + XGB + CatBoost, each with n_estimators=3000 on 100% of train |
| Blend | Ridge stacking weights from Phase 9 CV |
| Post-process | clip to [0, 1] — demand is naturally bounded |
| Output | `submissions/final_submission.csv` (41 778 × 2) |

In [ ]:
# 14.0 — Load features + best CV artefacts
print('Loading engineered features...')
train_feat = pd.read_csv(f'{OUT}/features/train_features.csv')
test_feat  = pd.read_csv(f'{OUT}/features/test_features.csv')
sub_tmpl   = pd.read_csv(f'{BASE}/sample_submission.csv')

FEAT_COLS  = [c for c in train_feat.columns if c not in ('Index','demand')]
Xfull      = train_feat[FEAT_COLS].copy()
yfull      = train_feat['demand'].values
Xtest_full = test_feat[FEAT_COLS].copy()
test_idx   = test_feat['Index'].values

# CatBoost integer-typed columns
CAT_COLS = [c for c in ['geo_cluster','geohash5_enc','geohash4_enc',
                        'time_of_day','day_of_week','temp_binned'] if c in FEAT_COLS]
for c in CAT_COLS:
    Xfull[c]      = Xfull[c].astype(int)
    Xtest_full[c] = Xtest_full[c].astype(int)

# Residual null safety
Xfull      = Xfull.fillna(Xfull.median())
Xtest_full = Xtest_full.fillna(Xfull.median())

# Load best CV stacking weights (Ridge trained on 5-fold OOF)
stk_v1 = joblib.load(f'{OUT}/models/stacking.pkl')
ridge_w = stk_v1['ridge'].coef_          # [lgb_w, xgb_w, cat_w]
ridge_i = stk_v1['ridge'].intercept_
oof_r2  = stk_v1['r2_meta']              # CV R² from Phase 9

print(f'Train features : {Xfull.shape}')
print(f'Test  features : {Xtest_full.shape}')
print(f'Feature count  : {len(FEAT_COLS)}')
print(f'CatBoost cats  : {CAT_COLS}')
print(f'Nulls train/test: {Xfull.isnull().sum().sum()} / {Xtest_full.isnull().sum().sum()}')
print(f'CV Ridge weights: LGB={ridge_w[0]:.4f}  XGB={ridge_w[1]:.4f}  CAT={ridge_w[2]:.4f}  intercept={ridge_i:.5f}')
print(f'Best CV R²      : {oof_r2:.6f}  →  competition score: {100*oof_r2:.4f}')

In [ ]:
# 14.1 — Load best hyperparameters; fix n_estimators for full-data training
#         (smoke-test params have n_estimators=120; override to 3000)
import copy

def fix_params(raw, model):
    p = copy.deepcopy(raw)
    if model == 'lgb':
        p['n_estimators'] = 3000
        p.update(verbose=-1, n_jobs=-1, random_state=SEED)
    elif model == 'xgb':
        p['n_estimators'] = 3000
        # remove early_stopping_rounds — no eval_set on full data
        p.pop('early_stopping_rounds', None)
        p.update(tree_method='hist', random_state=SEED, n_jobs=-1)
    elif model == 'cat':
        p['iterations'] = 3000
        p.update(random_seed=SEED, eval_metric='RMSE', verbose=False)
        p.pop('od_type', None); p.pop('od_wait', None)  # no early stop on full data
    return p

p_lgb = fix_params(json.load(open(f'{OUT}/models/best_lgb_params.json')), 'lgb')
p_xgb = fix_params(json.load(open(f'{OUT}/models/best_xgb_params.json')), 'xgb')
p_cat = fix_params(json.load(open(f'{OUT}/models/best_cat_params.json')), 'cat')

print('LGB params :', {k:v for k,v in p_lgb.items() if k not in ('n_jobs','verbose','random_state')})
print('XGB params :', {k:v for k,v in p_xgb.items() if k not in ('n_jobs','tree_method','random_state')})
print('CAT params :', {k:v for k,v in p_cat.items() if k not in ('random_seed','eval_metric','verbose')})

In [ ]:
# 14.2 — Train LightGBM on full training data
print('Training LightGBM (3000 trees, full data)...')
t0 = time.time()
final_lgb = lgb.LGBMRegressor(**p_lgb)
final_lgb.fit(Xfull, yfull)
pred_lgb = final_lgb.predict(Xtest_full)
print(f'  done in {time.time()-t0:.0f}s  pred: min={pred_lgb.min():.4f}  max={pred_lgb.max():.4f}  mean={pred_lgb.mean():.4f}')
joblib.dump(final_lgb, f'{OUT}/models/final_lgb.pkl')
print('  saved -> models/final_lgb.pkl')

In [ ]:
# 14.3 — Train XGBoost on full training data
print('Training XGBoost (3000 trees, full data)...')
t0 = time.time()
final_xgb = xgb.XGBRegressor(**p_xgb)
final_xgb.fit(Xfull, yfull, verbose=False)
pred_xgb = final_xgb.predict(Xtest_full)
print(f'  done in {time.time()-t0:.0f}s  pred: min={pred_xgb.min():.4f}  max={pred_xgb.max():.4f}  mean={pred_xgb.mean():.4f}')
joblib.dump(final_xgb, f'{OUT}/models/final_xgb.pkl')
print('  saved -> models/final_xgb.pkl')

In [ ]:
# 14.4 — Train CatBoost on full training data
print('Training CatBoost (3000 iterations, full data)...')
t0 = time.time()
final_cat = CatBoostRegressor(**p_cat)
final_cat.fit(Xfull, yfull, cat_features=CAT_COLS)
pred_cat = final_cat.predict(Xtest_full)
print(f'  done in {time.time()-t0:.0f}s  pred: min={pred_cat.min():.4f}  max={pred_cat.max():.4f}  mean={pred_cat.mean():.4f}')
joblib.dump(final_cat, f'{OUT}/models/final_cat.pkl')
print('  saved -> models/final_cat.pkl')

In [ ]:
# 14.5 — Ridge blend + post-processing
# Stack predictions and apply CV-derived Ridge weights
S_final = np.column_stack([pred_lgb, pred_xgb, pred_cat])
pred_blend = S_final @ ridge_w + ridge_i           # Ridge linear combination

# No log1p was applied — demand was predicted in original space
# Clip to valid demand range [0, 1]
pred_final = np.clip(pred_blend, 0.0, 1.0)

print(f'Raw blend stats  : min={pred_blend.min():.6f}  max={pred_blend.max():.6f}  '
      f'mean={pred_blend.mean():.6f}  std={pred_blend.std():.6f}')
print(f'Clipped <0       : {(pred_blend < 0).sum():5d}  ({100*(pred_blend<0).mean():.2f}%)')
print(f'Clipped >1       : {(pred_blend > 1).sum():5d}  ({100*(pred_blend>1).mean():.2f}%)')
print(f'Final pred stats : min={pred_final.min():.6f}  max={pred_final.max():.6f}  '
      f'mean={pred_final.mean():.6f}  std={pred_final.std():.6f}')

In [ ]:
# 14.6 — Full sanity checks
print('=' * 60)
print('SANITY CHECKS')
print('=' * 60)

checks = []
def chk(name, cond, detail=''):
    status = '✓ PASS' if cond else '✗ FAIL'
    checks.append(cond)
    print(f'  {status}  {name}  {detail}')

chk('Count = 41778',      len(pred_final) == 41778,         f'got {len(pred_final)}')
chk('No nulls',           ~np.isnan(pred_final).any(),      f'nulls: {np.isnan(pred_final).sum()}')
chk('Min >= 0',           pred_final.min() >= 0,            f'min={pred_final.min():.6f}')
chk('Max <= 1',           pred_final.max() <= 1.0,          f'max={pred_final.max():.6f}')
chk('No infs',            ~np.isinf(pred_final).any(),      f'infs: {np.isinf(pred_final).sum()}')

# distribution similarity
train_mean, train_std = yfull.mean(), yfull.std()
pred_mean,  pred_std  = pred_final.mean(), pred_final.std()
# test covers hours 2-13 only (mid-day); train covers all 24h including low-demand overnight.
# compute hour-distribution-weighted expected mean for a fair comparison
_test_raw    = pd.read_csv(f'{BASE}/test.csv')
_test_hours  = _test_raw['timestamp'].str.split(':').str[0].astype(int)
_train_raw2  = pd.read_csv(f'{BASE}/train.csv')
_train_raw2['_h'] = _train_raw2['timestamp'].str.split(':').str[0].astype(int)
_hr_mean     = _train_raw2.groupby('_h')['demand'].mean()
_test_h_dist = _test_hours.value_counts(normalize=True)
expected_mean= (_test_h_dist * _hr_mean.reindex(_test_h_dist.index).fillna(train_mean)).sum()
mean_ok = abs(pred_mean - expected_mean) < 0.03  # compare to hour-adjusted expected mean
std_ok  = abs(pred_std  - train_std)  < 0.04
chk('Mean ~ hour-adj expected', mean_ok, f'pred={pred_mean:.4f}  expected={expected_mean:.4f}  train_raw={train_mean:.4f}  (test is hours 2-13 only)')
chk('Std  similar to train', std_ok,  f'pred={pred_std:.4f}   train={train_std:.4f}   diff={abs(pred_std-train_std):.4f}')

# index alignment
chk('Index matches sample_sub', list(test_idx[:5]) == list(sub_tmpl['Index'][:5].values), f'first 5: {test_idx[:5].tolist()}')

print('=' * 60)
print(f'  Passed: {sum(checks)}/{len(checks)}')
if not all(checks): print('  *** REVIEW FAILED CHECKS BEFORE SUBMITTING ***')

# Distribution comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(yfull,      bins=80, alpha=0.6, color='steelblue', label=f'Train actual (μ={train_mean:.3f})')
axes[0].hist(pred_final, bins=80, alpha=0.6, color='coral',     label=f'Test pred   (μ={pred_mean:.3f})')
axes[0].set_title('Demand Distribution: Train vs Final Predictions')
axes[0].legend()
axes[1].hist(np.log1p(yfull),      bins=80, alpha=0.6, color='steelblue', label='Train actual')
axes[1].hist(np.log1p(pred_final), bins=80, alpha=0.6, color='coral',     label='Test pred')
axes[1].set_title('log1p(demand): Train vs Final Predictions')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{OUT}/submissions/final_distribution_check.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 14.7 — Build and save final_submission.csv
final_sub = pd.DataFrame({'Index': test_idx, 'demand': pred_final})

# shape check
assert final_sub.shape == (41778, 2), f'Wrong shape: {final_sub.shape}'
assert list(final_sub.columns) == ['Index', 'demand'], f'Wrong columns: {final_sub.columns.tolist()}'
assert final_sub.isnull().sum().sum() == 0, 'Nulls found!'

sub_path = f'{OUT}/submissions/final_submission.csv'
final_sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
print(f'Shape: {final_sub.shape}')
print(f'dtypes:\n{final_sub.dtypes.to_string()}')
print(f'Nulls : {final_sub.isnull().sum().sum()}')
print(f'\nFirst 10 rows:')
print(final_sub.head(10).to_string(index=False))
print(f'\nLast 10 rows:')
print(final_sub.tail(10).to_string(index=False))
print(f'\nSubmission stats:')
print(final_sub['demand'].describe().to_string())

In [ ]:
# 14.8 — SHAP feature importance from full-data LGB
import shap
print('Computing SHAP values (sample=3000 rows)...')
shap_samp = Xfull.sample(min(3000, len(Xfull)), random_state=SEED)
explainer  = shap.TreeExplainer(final_lgb)
shap_vals  = explainer.shap_values(shap_samp)
shap_imp   = pd.DataFrame({'feature': FEAT_COLS,
                           'shap_importance': np.abs(shap_vals).mean(axis=0)})\
               .sort_values('shap_importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(data=shap_imp.head(20), x='shap_importance', y='feature',
            palette='viridis', ax=axes[0])
axes[0].set_title('Final Model — Top 20 SHAP Feature Importance')
axes[0].set_xlabel('mean(|SHAP|)')

shap.summary_plot(shap_vals, shap_samp, plot_type='dot',
                  max_display=15, show=False)
axes[1].set_title('SHAP Dot Plot (top 15)')
plt.tight_layout()
plt.savefig(f'{OUT}/submissions/final_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 10 SHAP features:')
print(shap_imp.head(10).to_string(index=False))
shap_imp.to_csv(f'{OUT}/features/final_shap_importance.csv', index=False)

In [ ]:
# 14.9 — Final Report
print()
print('╔' + '═'*62 + '╗')
print('║' + ' FLIPKART TRAFFIC DEMAND PREDICTION — FINAL REPORT '.center(62) + '║')
print('╠' + '═'*62 + '╣')

print('║ PERFORMANCE'.ljust(63) + '║')
print(f'║   Best CV R²          : {oof_r2:.6f}'.ljust(63) + '║')
print(f'║   Competition score   : {100*oof_r2:.4f}  (target > 95.0)'.ljust(63) + '║')
target_met = '✓ TARGET MET' if oof_r2 >= 0.95 else '✗ below target'
print(f'║   Status              : {target_met}'.ljust(63) + '║')

print('╠' + '═'*62 + '╣')
print('║ FINAL ENSEMBLE (full-data retrain)'.ljust(63) + '║')
print(f'║   LightGBM  weight={ridge_w[0]:.4f}   n_estimators=3000'.ljust(63) + '║')
print(f'║   XGBoost   weight={ridge_w[1]:.4f}   n_estimators=3000'.ljust(63) + '║')
print(f'║   CatBoost  weight={ridge_w[2]:.4f}   iterations=3000'.ljust(63) + '║')
print(f'║   Intercept           : {ridge_i:.6f}'.ljust(63) + '║')
print(f'║   Blending            : Ridge meta-learner (5-fold CV)'.ljust(63) + '║')

print('╠' + '═'*62 + '╣')
print('║ TOP 10 FEATURES (SHAP, final LGB)'.ljust(63) + '║')
for i, row in shap_imp.head(10).iterrows():
    line = f'║   {i+1:2d}. {row["feature"]:<30} {row["shap_importance"]:.5f}'
    print(line.ljust(63) + '║')

print('╠' + '═'*62 + '╣')
print('║ KEY FEATURE ENGINEERING DECISIONS'.ljust(63) + '║')
decisions = [
    'OOF target encoding (geo×slot, geo×hour) — #1 signal',
    'Geohash decode → lat/lon (spatial smoothing)',
    'Cyclical time encoding (sin/cos) for hour/slot',
    'KMeans geo clusters (k=20, best silhouette)',
    'Road capacity score (lanes × large-vehicle flag)',
    'Geohash neighbor mean/max demand (context)',
    'Temperature imputation by geohash median',
    'RoadType imputed from geohash mode (79.6% stable)',
    'Frequency encoding (geohash, geohash×hour)',
]
for d in decisions:
    print(f'║   • {d}'.ljust(63) + '║')

print('╠' + '═'*62 + '╣')
print('║ SUBMISSION FILE'.ljust(63) + '║')
print(f'║   Path   : submissions/final_submission.csv'.ljust(63) + '║')
print(f'║   Shape  : {final_sub.shape[0]} rows × {final_sub.shape[1]} cols'.ljust(63) + '║')
print(f'║   Demand : min={pred_final.min():.4f}  max={pred_final.max():.4f}  '
      f'mean={pred_final.mean():.4f}'.ljust(43) + '║')
print(f'║   Nulls  : {final_sub.isnull().sum().sum()}'.ljust(63) + '║')
print('╚' + '═'*62 + '╝')

---
## Phase 15 — Custom Neural Architecture & CV-Leakage Fix

### Root cause of 91.24 vs 96.22 CV gap
Our random-5-fold CV mixed **day=48 and day=49 rows** in both train and
validation. The OOF target-encoded features (`te_geohash_hour` etc.) could
see demand from the *same geohash at the same hour* on a different day inside
the fold bank — inflating CV by ~5 R² points. The leaderboard correctly
evaluates on unseen day=49 only.

### Fix
| Problem | Solution |
|---------|----------|
| Leaky random-fold TE | Cross-day TE: stats computed on **day=48 only**, applied to day=49/test |
| Missing inter-day signal | **Yesterday's demand** `(geohash, slot)` → day=48 lookup |
| GBM memorises demand | **Custom MLP + geohash embeddings** — learns location structure, not demand values |
| Wrong CV strategy | **Temporal split**: train=day48, val=day49-from-train |

### Pipeline
```
15.0  Day-split analysis + yesterday-demand computation
15.1  V3 feature matrix (cross-day TE + yesterday demand)
15.2  DemandNet — PyTorch entity-embedding MLP
15.3  Neural network training  (temporal split → full-data retrain)
15.4  LGB / XGB / CatBoost retrain on V3 features  (temporal split)
15.5  4-model ensemble  (Neural + LGB + XGB + CAT)
15.6  Final submission v3
```


In [ ]:
# 15.0 — Day-split analysis + yesterday-demand lookup
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

train_raw = pd.read_csv(f'{BASE}/train.csv')
test_raw  = pd.read_csv(f'{BASE}/test.csv')

for df in [train_raw, test_raw]:
    parts = df['timestamp'].str.split(':')
    df['hour']      = parts.str[0].astype(int)
    df['minute']    = parts.str[1].astype(int)
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15

day48 = train_raw[train_raw['day'] == 48].copy()
day49 = train_raw[train_raw['day'] == 49].copy()   # val split mirror

print(f'day=48 rows : {len(day48):,}   hours {day48["hour"].min()}–{day48["hour"].max()}')
print(f'day=49 rows : {len(day49):,}   hours {day49["hour"].min()}–{day49["hour"].max()}')
print(f'test   rows : {len(test_raw):,}   hours {test_raw["hour"].min()}–{test_raw["hour"].max()}')

# ── yesterday-demand lookup (day48 stats → applied to day49/test) ──────
# For each (geohash, time_slot): mean demand from day=48
yd_geo_slot = (day48.groupby(['geohash', 'time_slot'])['demand']
               .mean().rename('yd_geo_slot'))
yd_geo      = day48.groupby('geohash')['demand'].mean().rename('yd_geo')
yd_slot     = day48.groupby('time_slot')['demand'].mean().rename('yd_slot')
yd_global   = day48['demand'].mean()

def yesterday_demand(geohash_s, slot_s):
    """Cross-day lookup: day=48 (geohash, slot) mean demand. Cascade fallback."""
    idx = pd.MultiIndex.from_arrays([geohash_s, slot_s.astype(int)])
    v   = yd_geo_slot.reindex(idx).values
    geo_fb = geohash_s.map(yd_geo).values
    slot_fb= pd.Series(slot_s.astype(int)).map(yd_slot).values
    v = np.where(np.isnan(v), geo_fb,  v)
    v = np.where(np.isnan(v), slot_fb, v)
    v = np.where(np.isnan(v), yd_global, v)
    return v

# sanity: predictability on day49 val rows
from sklearn.metrics import r2_score as _r2
yd_check = yesterday_demand(day49['geohash'], day49['time_slot'])
print(f'\nYesterday-demand feature alone → R² on day49 val: {_r2(day49["demand"], yd_check):.5f}')
print('(day49-val covers hours 0-2 only; test is hours 2-13 → stronger cross-day signal expected in test)')

In [ ]:
# 15.1 — V3 feature matrix (fixed)
import pygeohash as pgh, copy

# ── pre-enrich day48 with columns cd_te will need ──────────────────────────
day48['time_of_day'] = pd.cut(day48['hour'],[-1,5,11,16,20,24],
                              labels=[0,1,2,3,4]).astype(float)
day48['RoadType']    = day48['RoadType'].fillna('Residential')
day48['Weather']     = day48['Weather'].fillna('Sunny')

def build_v3(df_raw, is_train=True, geo2id=None, road2id=None, wx2id=None):
    d = df_raw.copy()

    # ── time ─────────────────────────────────────────────────────────────
    d['hour_sin']    = np.sin(2*np.pi*d['hour']/24)
    d['hour_cos']    = np.cos(2*np.pi*d['hour']/24)
    d['minute_sin']  = np.sin(2*np.pi*d['minute']/60)
    d['minute_cos']  = np.cos(2*np.pi*d['minute']/60)
    d['slot_sin']    = np.sin(2*np.pi*d['time_slot']/96)
    d['slot_cos']    = np.cos(2*np.pi*d['time_slot']/96)
    d['is_peak']     = d['hour'].isin([7,8,9,17,18,19,20]).astype(int)
    d['is_night']    = d['hour'].isin([23,0,1,2,3,4,5]).astype(int)
    d['time_of_day'] = pd.cut(d['hour'],[-1,5,11,16,20,24],
                              labels=[0,1,2,3,4]).astype(float)

    # ── geospatial ───────────────────────────────────────────────────────
    all_gh = d['geohash'].unique()
    coords = {}
    for gh in all_gh:
        r = pgh.decode_exactly(gh)
        coords[gh] = (r.latitude, r.longitude)
    d['lat'] = d['geohash'].map(lambda g: coords[g][0])
    d['lon'] = d['geohash'].map(lambda g: coords[g][1])
    clat = day48['geohash'].map(lambda g: pgh.decode_exactly(g).latitude).mean()
    clon = day48['geohash'].map(lambda g: pgh.decode_exactly(g).longitude).mean()
    def hav(la, lo):
        R=6371; p=np.pi/180
        a = 0.5 - np.cos((clat-la)*p)/2 + np.cos(la*p)*np.cos(clat*p)*(1-np.cos((clon-lo)*p))/2
        return 2*R*np.arcsin(np.sqrt(a))
    d['dist_center'] = hav(d['lat'].values, d['lon'].values)

    # ── road / infra ──────────────────────────────────────────────────────
    rt_mode = (day48.dropna(subset=['RoadType'])
               .groupby('geohash')['RoadType']
               .agg(lambda s: s.mode().iat[0]))
    d['RoadType']     = d['RoadType'].fillna(d['geohash'].map(rt_mode)).fillna('Residential')
    wx_mode = (day48.dropna(subset=['Weather'])
               .groupby('geohash')['Weather']
               .agg(lambda s: s.mode().iat[0]))
    d['Weather']      = d['Weather'].fillna(d['geohash'].map(wx_mode)).fillna('Sunny')
    d['LargeVeh_bin'] = (d['LargeVehicles'] == 'Allowed').astype(int)
    d['Landmarks_bin']= (d['Landmarks'] == 'Yes').astype(int)
    road_ord = {'Residential':0,'Street':1,'Highway':2}
    d['road_ord']     = d['RoadType'].map(road_ord).fillna(0).astype(int)
    d['road_cap']     = d['NumberofLanes'] * (1 + d['LargeVeh_bin'])

    # ── weather / temperature ─────────────────────────────────────────────
    geo_temp  = day48.groupby('geohash')['Temperature'].median()
    glob_temp = day48['Temperature'].median()
    d['Temperature'] = d['Temperature'].fillna(d['geohash'].map(geo_temp)).fillna(glob_temp)
    wx_sev = {'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}
    d['wx_sev']  = d['Weather'].map(wx_sev).fillna(0).astype(int)
    d['temp_sq'] = d['Temperature']**2

    # ── CROSS-DAY target encoding (merge-based, no leakage) ──────────────
    # Statistics computed from day=48 ONLY and merged into d.
    def cd_te(gcols, agg_fn, col):
        src = day48.copy()
        stat = (src.groupby(gcols, observed=True)['demand']
                .agg(agg_fn).reset_index()
                .rename(columns={'demand': col}))
        fb  = float(src['demand'].agg(agg_fn))
        tmp = d[gcols].copy()
        # cast time_of_day to same dtype if needed
        if 'time_of_day' in gcols:
            tmp['time_of_day'] = tmp['time_of_day'].astype(float)
            stat['time_of_day'] = stat['time_of_day'].astype(float)
        merged = tmp.merge(stat, on=gcols, how='left')
        d[col] = merged[col].fillna(fb).values

    cd_te(['geohash'],                      'mean',   'cd_geo_mean')
    cd_te(['geohash'],                      'max',    'cd_geo_max')
    cd_te(['geohash'],                      'std',    'cd_geo_std')
    cd_te(['geohash','time_slot'],          'mean',   'cd_geo_slot_mean')
    cd_te(['geohash','hour'],               'mean',   'cd_geo_hour_mean')
    cd_te(['geohash','time_of_day'],        'mean',   'cd_geo_tod_mean')
    cd_te(['RoadType','hour'],              'mean',   'cd_road_hour_mean')
    cd_te(['geohash','hour'],               'median', 'cd_geo_hour_med')
    cd_te(['geohash','hour'],               'std',    'cd_geo_hour_std')
    d['cd_geo_std']      = d['cd_geo_std'].fillna(0)
    d['cd_geo_hour_std'] = d['cd_geo_hour_std'].fillna(0)

    # ── Yesterday demand ─────────────────────────────────────────────────
    d['yd']       = yesterday_demand(d['geohash'], d['time_slot'])
    d['yd_lag4']  = yesterday_demand(d['geohash'], d['time_slot'] - 4)
    d['yd_lag8']  = yesterday_demand(d['geohash'], d['time_slot'] - 8)
    d['yd_lead4'] = yesterday_demand(d['geohash'], d['time_slot'] + 4)
    d['yd_ratio'] = d['yd'] / (d['cd_geo_mean'] + 1e-6)

    # ── Frequency encoding ───────────────────────────────────────────────
    d['geo_freq'] = d['geohash'].map(day48['geohash'].value_counts()).fillna(0)

    # ── Integer IDs for neural network embeddings ─────────────────────────
    all_geo_ids = pd.concat([day48['geohash'], test_raw['geohash']]).unique()
    if geo2id is None:
        geo2id = {g: i+1 for i, g in enumerate(sorted(all_geo_ids))}
    if road2id is None:
        road2id = {'Residential':1,'Street':2,'Highway':3}
    if wx2id is None:
        wx2id = {'Sunny':1,'Rainy':2,'Foggy':3,'Snowy':4}
    d['geo_id']  = d['geohash'].map(geo2id).fillna(0).astype(int)
    d['road_id'] = d['RoadType'].map(road2id).fillna(1).astype(int)
    d['wx_id']   = d['Weather'].map(wx2id).fillna(1).astype(int)

    return d, geo2id, road2id, wx2id

# Build V3 for all splits
d48_v3, geo2id, road2id, wx2id = build_v3(day48,    is_train=True)
d49_v3, _,      _,       _     = build_v3(day49,    is_train=False,
                                           geo2id=geo2id, road2id=road2id, wx2id=wx2id)
te_v3,  _,      _,       _     = build_v3(test_raw, is_train=False,
                                           geo2id=geo2id, road2id=road2id, wx2id=wx2id)
full_v3 = pd.concat([d48_v3, d49_v3], ignore_index=True)

GBM_FEATS = [
    'hour','minute','time_slot','hour_sin','hour_cos','minute_sin','minute_cos',
    'slot_sin','slot_cos','is_peak','is_night','time_of_day',
    'lat','lon','dist_center',
    'road_ord','NumberofLanes','LargeVeh_bin','Landmarks_bin','road_cap',
    'wx_sev','Temperature','temp_sq',
    'cd_geo_mean','cd_geo_max','cd_geo_std',
    'cd_geo_slot_mean','cd_geo_hour_mean','cd_geo_tod_mean',
    'cd_road_hour_mean','cd_geo_hour_med','cd_geo_hour_std',
    'yd','yd_lag4','yd_lag8','yd_lead4','yd_ratio','geo_freq',
]
NN_NUMERIC = [
    'lat','lon','dist_center','hour_sin','hour_cos','minute_sin','minute_cos',
    'slot_sin','slot_cos','is_peak','is_night',
    'Temperature','temp_sq','NumberofLanes','LargeVeh_bin','Landmarks_bin','road_cap',
    'wx_sev','cd_geo_mean','cd_geo_slot_mean','cd_geo_hour_mean',
    'yd','yd_lag4','yd_lag8','yd_lead4','yd_ratio','geo_freq',
]

def get_arrays(df):
    X_num  = df[NN_NUMERIC].fillna(0).values.astype(np.float32)
    geo_id = df['geo_id'].values.astype(np.int64)
    slot   = df['time_slot'].values.astype(np.int64)
    road   = df['road_id'].values.astype(np.int64)
    wx     = df['wx_id'].values.astype(np.int64)
    y_arr  = df['demand'].values.astype(np.float32) if 'demand' in df.columns else None
    return geo_id, slot, road, wx, X_num, y_arr

print(f'V3 features: {len(GBM_FEATS)} GBM, {len(NN_NUMERIC)} NN-numeric')
print(f'Unique geohash IDs: {len(geo2id)}  (max_id={max(geo2id.values())})')
print(f'day48 train: {len(d48_v3):,}   day49 val: {len(d49_v3):,}   test: {len(te_v3):,}')
print('Sample yd values:', d48_v3['yd'].head(5).values.round(4))


In [ ]:
# 15.2 — DemandNet: entity-embedding MLP with residual blocks

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim, bias=False),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim, bias=False),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class DemandNet(nn.Module):
    """
    Entity-embedding MLP for traffic demand prediction.
    Learns a 64-dim vector for each geohash, 32-dim for time slot,
    8-dim for road type and weather — generalises across days
    without memorising demand values.
    """
    def __init__(self, n_geo, n_numeric, slot_dim=32, geo_dim=64,
                 road_dim=8, wx_dim=8, hidden=512, dropout=0.15):
        super().__init__()
        self.geo_emb  = nn.Embedding(n_geo+1,  geo_dim,  padding_idx=0)
        self.slot_emb = nn.Embedding(96,        slot_dim)
        self.road_emb = nn.Embedding(4,         road_dim, padding_idx=0)
        self.wx_emb   = nn.Embedding(5,         wx_dim,   padding_idx=0)

        in_dim = geo_dim + slot_dim + road_dim + wx_dim + n_numeric
        self.input_norm = nn.BatchNorm1d(n_numeric)

        self.entry = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
        )
        self.res1  = ResBlock(hidden, dropout)
        self.res2  = ResBlock(hidden, dropout)
        self.down1 = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.BatchNorm1d(hidden//2),
            nn.GELU(),
        )
        self.res3  = ResBlock(hidden//2, dropout)
        self.res4  = ResBlock(hidden//2, dropout)
        self.down2 = nn.Sequential(
            nn.Linear(hidden//2, hidden//4),
            nn.BatchNorm1d(hidden//4),
            nn.GELU(),
        )
        self.res5  = ResBlock(hidden//4, dropout)
        self.head  = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden//4, 1),
            nn.Sigmoid(),        # demand ∈ [0, 1]
        )

    def forward(self, geo, slot, road, wx, num):
        x = torch.cat([
            self.geo_emb(geo),
            self.slot_emb(slot),
            self.road_emb(road),
            self.wx_emb(wx),
            self.input_norm(num),
        ], dim=1)
        x = self.entry(x)
        x = self.res2(self.res1(x))
        x = self.down1(x)
        x = self.res4(self.res3(x))
        x = self.down2(x)
        x = self.res5(x)
        return self.head(x).squeeze(-1)

class DemandDataset(Dataset):
    def __init__(self, geo, slot, road, wx, num, y=None):
        self.geo, self.slot, self.road, self.wx = (
            torch.tensor(geo), torch.tensor(slot),
            torch.tensor(road), torch.tensor(wx))
        self.num = torch.tensor(num)
        self.y   = torch.tensor(y) if y is not None else None
    def __len__(self):  return len(self.geo)
    def __getitem__(self, i):
        out = (self.geo[i], self.slot[i], self.road[i], self.wx[i], self.num[i])
        return out + (self.y[i],) if self.y is not None else out

# Force CPU: MPS segfaults with OneCycleLR in PyTorch 2.x on macOS
device = torch.device('cpu')
# (swap to mps/cuda once PyTorch MPS stability improves)
print(f'Device: {device}')
n_geo    = max(geo2id.values())
n_numeric= len(NN_NUMERIC)
# Quick architecture sanity check
_net = DemandNet(n_geo, n_numeric).to(device)
_dummy_out = _net(
    torch.zeros(4, dtype=torch.long).to(device),
    torch.zeros(4, dtype=torch.long).to(device),
    torch.ones(4,  dtype=torch.long).to(device),
    torch.ones(4,  dtype=torch.long).to(device),
    torch.zeros(4, n_numeric).to(device),
)
params = sum(p.numel() for p in _net.parameters())
print(f'DemandNet params: {params:,}  output shape: {_dummy_out.shape}')
del _net, _dummy_out

In [ ]:
# 15.3 — DemandNet training  (temporal split: train=day48, val=day49)

EPOCHS    = 200
BATCH     = 4096
PATIENCE  = 20
LR        = 3e-3
NN_CKPT   = f'{OUT}/models/demand_net.pt'

# datasets
tr_geo,tr_slot,tr_road,tr_wx,tr_num,tr_y = get_arrays(d48_v3)
va_geo,va_slot,va_road,va_wx,va_num,va_y = get_arrays(d49_v3)

# normalise numerics on day=48 stats
num_mean = tr_num.mean(axis=0, keepdims=True)
num_std  = tr_num.std(axis=0,  keepdims=True) + 1e-8
tr_num_n = ((tr_num - num_mean) / num_std).astype(np.float32)
va_num_n = ((va_num - num_mean) / num_std).astype(np.float32)

tr_ds = DemandDataset(tr_geo,tr_slot,tr_road,tr_wx,tr_num_n,tr_y)
va_ds = DemandDataset(va_geo,va_slot,va_road,va_wx,va_num_n,va_y)
tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=False)
va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=False)

torch.manual_seed(SEED)
net   = DemandNet(n_geo, n_numeric).to(device)
opt   = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=LR, epochs=EPOCHS,
    steps_per_epoch=len(tr_dl), pct_start=0.1, div_factor=10)
loss_fn = nn.HuberLoss(delta=0.1)   # robust to Highway outliers

best_val_r2, best_ep, patience_cnt = -np.inf, 0, 0

for ep in range(1, EPOCHS + 1):
    # ── train ──────────────────────────────────────────────────────────
    net.train()
    tr_loss = 0.0
    for batch in tr_dl:
        g, sl, ro, wx, num, y = [b.to(device) for b in batch]
        opt.zero_grad()
        pred = net(g, sl, ro, wx, num)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step(); sched.step()
        tr_loss += loss.item() * len(y)
    tr_loss /= len(tr_ds)

    # ── validate ───────────────────────────────────────────────────────
    net.eval()
    preds_va = []
    with torch.no_grad():
        for batch in va_dl:
            g, sl, ro, wx, num, _ = [b.to(device) for b in batch]
            preds_va.append(net(g, sl, ro, wx, num).cpu().numpy())
    preds_va = np.concatenate(preds_va)
    val_r2   = r2_score(va_y, preds_va)

    if ep % 10 == 0 or ep <= 5:
        print(f'  ep {ep:3d}  tr_loss={tr_loss:.5f}  val_R²={val_r2:.5f}')

    if val_r2 > best_val_r2:
        best_val_r2, best_ep = val_r2, ep
        patience_cnt = 0
        torch.save({'state': net.state_dict(),
                    'num_mean': num_mean, 'num_std': num_std,
                    'ep': ep, 'val_r2': val_r2}, NN_CKPT)
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'  Early stop at ep {ep} (best ep={best_ep})')
            break

print(f'\nBest val R² = {best_val_r2:.5f} at epoch {best_ep}')
print(f'Checkpoint saved: {NN_CKPT}')

In [ ]:
# 15.4 — DemandNet full-data retrain (day48+day49) + test predictions
#         Retrain for best_ep epochs on all available training data.

ckpt = torch.load(NN_CKPT, map_location='cpu')
best_ep_nn = ckpt['ep']
print(f'Retraining for {best_ep_nn} epochs on full train (day48+day49)...')

fu_geo, fu_slot, fu_road, fu_wx, fu_num, fu_y = get_arrays(full_v3)
te_geo, te_slot, te_road, te_wx, te_num, _    = get_arrays(te_v3)

# normalise on full train stats
fn_mean = fu_num.mean(axis=0, keepdims=True)
fn_std  = fu_num.std(axis=0,  keepdims=True) + 1e-8
fu_num_n= ((fu_num - fn_mean) / fn_std).astype(np.float32)
te_num_n= ((te_num - fn_mean) / fn_std).astype(np.float32)

fu_ds   = DemandDataset(fu_geo,fu_slot,fu_road,fu_wx,fu_num_n,fu_y)
te_ds   = DemandDataset(te_geo,te_slot,te_road,te_wx,te_num_n)
fu_dl   = DataLoader(fu_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
te_dl   = DataLoader(te_ds, batch_size=BATCH, shuffle=False, num_workers=0)

torch.manual_seed(SEED)
net_full = DemandNet(n_geo, n_numeric).to(device)
opt_full = torch.optim.AdamW(net_full.parameters(), lr=LR, weight_decay=1e-4)
sch_full = torch.optim.lr_scheduler.OneCycleLR(
    opt_full, max_lr=LR, epochs=best_ep_nn,
    steps_per_epoch=len(fu_dl), pct_start=0.1, div_factor=10)

for ep in range(1, best_ep_nn + 1):
    net_full.train()
    for batch in fu_dl:
        g,sl,ro,wx,num,y = [b.to(device) for b in batch]
        opt_full.zero_grad()
        loss_fn(net_full(g,sl,ro,wx,num), y).backward()
        nn.utils.clip_grad_norm_(net_full.parameters(), 1.0)
        opt_full.step(); sch_full.step()
    if ep % 20 == 0 or ep == best_ep_nn:
        print(f'  retrain ep {ep}/{best_ep_nn}')

net_full.eval()
nn_test_preds = []
with torch.no_grad():
    for batch in te_dl:
        g,sl,ro,wx,num = [b.to(device) for b in batch]
        nn_test_preds.append(net_full(g,sl,ro,wx,num).cpu().numpy())
nn_test = np.clip(np.concatenate(nn_test_preds), 0, 1)

# in-sample on full train (sanity)
nn_full_preds = []
with torch.no_grad():
    for batch in fu_dl:
        g,sl,ro,wx,num,_ = [b.to(device) for b in batch]
        nn_full_preds.append(net_full(g,sl,ro,wx,num).cpu().numpy())
nn_full_oof = np.clip(np.concatenate(nn_full_preds), 0, 1)

torch.save({'state': net_full.state_dict(),
            'num_mean': fn_mean, 'num_std': fn_std}, f'{OUT}/models/demand_net_full.pt')
np.save(f'{OUT}/models/nn_test_preds.npy', nn_test)
print(f'NN test preds: min={nn_test.min():.4f}  max={nn_test.max():.4f}  mean={nn_test.mean():.4f}')

In [ ]:
# 15.5 — LGB / XGB / CatBoost retrained on V3 features
#         Temporal split: day48 train → day49 val (honest, mirrors leaderboard)
#         Final models trained on full data (day48+day49) for test prediction.

X48  = d48_v3[GBM_FEATS].fillna(0);  y48  = d48_v3['demand'].values
X49  = d49_v3[GBM_FEATS].fillna(0);  y49  = d49_v3['demand'].values
Xfull_v3 = full_v3[GBM_FEATS].fillna(0); yfull_v3 = full_v3['demand'].values
Xte_v3   = te_v3[GBM_FEATS].fillna(0)

CAT_V3 = []

# ── LGB ──
print('LGB (temporal split validation)...')
p_lgb_v3 = {**json.load(open(f'{OUT}/models/best_lgb_params.json')),
            'n_estimators':3000, 'verbose':-1, 'n_jobs':-1, 'random_state':SEED}
lgb_v3 = lgb.LGBMRegressor(**p_lgb_v3)
lgb_v3.fit(X48, y48, eval_set=[(X49, y49)],
           callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)])
val_lgb = lgb_v3.predict(X49)
print(f'  LGB day49-val R² = {r2_score(y49, val_lgb):.5f}  (best_iter={lgb_v3.best_iteration_})')

print('LGB final (full data)...')
p_lgb_final = {**p_lgb_v3, 'n_estimators': lgb_v3.best_iteration_ or 1000}
lgb_final_v3 = lgb.LGBMRegressor(**p_lgb_final)
lgb_final_v3.fit(Xfull_v3, yfull_v3)
pred_lgb_v3 = lgb_final_v3.predict(Xte_v3)
joblib.dump(lgb_final_v3, f'{OUT}/models/final_lgb_v3.pkl')

# ── XGB ──
print('XGB (temporal split validation)...')
p_xgb_v3 = {**json.load(open(f'{OUT}/models/best_xgb_params.json')),
            'n_estimators':3000, 'tree_method':'hist',
            'random_state':SEED, 'n_jobs':-1,
            'early_stopping_rounds':150, 'eval_metric':'rmse'}
xgb_v3 = xgb.XGBRegressor(**p_xgb_v3)
xgb_v3.fit(X48, y48, eval_set=[(X49, y49)], verbose=False)
val_xgb = xgb_v3.predict(X49)
print(f'  XGB day49-val R² = {r2_score(y49, val_xgb):.5f}  (best_iter={xgb_v3.best_iteration})')

print('XGB final (full data)...')
p_xgb_final = {k:v for k,v in p_xgb_v3.items() if k not in ('early_stopping_rounds','eval_metric')}
p_xgb_final['n_estimators'] = xgb_v3.best_iteration or 1000
xgb_final_v3 = xgb.XGBRegressor(**p_xgb_final)
xgb_final_v3.fit(Xfull_v3, yfull_v3, verbose=False)
pred_xgb_v3 = xgb_final_v3.predict(Xte_v3)
joblib.dump(xgb_final_v3, f'{OUT}/models/final_xgb_v3.pkl')

# ── CatBoost ──
print('CatBoost (temporal split validation)...')
p_cat_v3 = {**json.load(open(f'{OUT}/models/best_cat_params.json')),
            'iterations':3000, 'random_seed':SEED, 'eval_metric':'RMSE',
            'od_type':'Iter', 'od_wait':150, 'verbose':False}
cat_v3 = CatBoostRegressor(**p_cat_v3)
cat_v3.fit(X48, y48, eval_set=(X49, y49), use_best_model=True)
val_cat = cat_v3.predict(X49)
print(f'  CAT day49-val R² = {r2_score(y49, val_cat):.5f}')

print('CatBoost final (full data)...')
p_cat_final = {**p_cat_v3, 'iterations': cat_v3.best_iteration_ or 1000}
p_cat_final.pop('od_type',None); p_cat_final.pop('od_wait',None)
cat_final_v3 = CatBoostRegressor(**p_cat_final)
cat_final_v3.fit(Xfull_v3, yfull_v3)
pred_cat_v3 = cat_final_v3.predict(Xte_v3)
joblib.dump(cat_final_v3, f'{OUT}/models/final_cat_v3.pkl')

print('\n=== Temporal-split validation R² (honest, matches leaderboard) ===')
for name,pv in [('LGB',val_lgb),('XGB',val_xgb),('CAT',val_cat)]:
    print(f'  {name}: {r2_score(y49,pv):.5f}')

In [ ]:
# 15.6 — 4-model ensemble: Neural Net + LGB + XGB + CatBoost
#         Weights optimised on day49 validation set (honest temporal holdout).

# val predictions from all 4 models
net_full.eval()
va_num_fn = ((va_num - fn_mean) / fn_std).astype(np.float32)   # normalise on full-train stats
va_ds_fn  = DemandDataset(va_geo,va_slot,va_road,va_wx,va_num_fn)
va_dl_fn  = DataLoader(va_ds_fn, batch_size=BATCH, shuffle=False, num_workers=0)
nn_val_preds = []
with torch.no_grad():
    for batch in va_dl_fn:
        g,sl,ro,wx,num = [b.to(device) for b in batch]
        nn_val_preds.append(net_full(g,sl,ro,wx,num).cpu().numpy())
nn_val = np.clip(np.concatenate(nn_val_preds), 0, 1)

S_val  = np.column_stack([nn_val,  val_lgb,  val_xgb,  val_cat])
S_test = np.column_stack([nn_test, pred_lgb_v3, pred_xgb_v3, pred_cat_v3])
names4 = ['NeuralNet', 'LGB', 'XGB', 'CatBoost']

# individual scores on day49 val
print('Individual day49-val R²:')
for n, pv in zip(names4, S_val.T):
    print(f'  {n:<12}: {r2_score(y49, pv):.5f}')

# Nelder-Mead optimal blend weights
def neg_r2_blend(w):
    w = np.clip(w, 0, None); s = w.sum()
    if s < 1e-9: return 0
    return -r2_score(y49, S_val @ (w / s))

best_r2_e, best_w_e = -np.inf, None
for _ in range(20):   # 20 random restarts for robustness
    w0 = np.random.dirichlet(np.ones(4))
    res = minimize(neg_r2_blend, w0, method='Nelder-Mead',
                   options={'xatol':1e-8,'fatol':1e-10,'maxiter':5000})
    w_try = np.clip(res.x,0,None); w_try /= w_try.sum()
    r2_try = r2_score(y49, S_val @ w_try)
    if r2_try > best_r2_e:
        best_r2_e, best_w_e = r2_try, w_try

print(f'\nOptimal blend weights (day49 val):')
for n, w in zip(names4, best_w_e):
    print(f'  {n:<12}: {w:.4f}')
print(f'Ensemble day49-val R² = {best_r2_e:.5f}  →  estimated score: {100*best_r2_e:.4f}')

# final test predictions
pred_v3_final = np.clip(S_test @ best_w_e, 0, 1)
np.save(f'{OUT}/models/test_preds_v3.npy', S_test)
joblib.dump({'weights': best_w_e, 'names': names4, 'val_r2': best_r2_e},
            f'{OUT}/models/stacking_v3.pkl')
print(f'Test pred: min={pred_v3_final.min():.5f}  max={pred_v3_final.max():.5f}  mean={pred_v3_final.mean():.5f}')

In [ ]:
# 15.7 — Final submission v3

sub_v3 = pd.DataFrame({'Index': test_raw['Index'].values, 'demand': pred_v3_final})
assert sub_v3.shape == (41778, 2),     f'Wrong shape: {sub_v3.shape}'
assert sub_v3.isnull().sum().sum()==0, 'Nulls!'
assert sub_v3['demand'].min() >= 0,    'Negative demand!'
assert sub_v3['demand'].max() <= 1,    'Demand > 1!'

sub_path = f'{OUT}/submissions/final_submission_v3.csv'
sub_v3.to_csv(sub_path, index=False)

# comparison table
print('╔' + '═'*62 + '╗')
print('║' + ' PHASE 15 FINAL REPORT '.center(62) + '║')
print('╠' + '═'*62 + '╣')
print('║ ROOT CAUSE'.ljust(63)+ '║')
print('║  Random-CV mixed days → OOF TE saw same geo×hour demand  ║')
print('║  across days → +5 R² overestimate. Fixed by temporal CV. ║')
print('╠' + '═'*62 + '╣')
print('║ VALIDATION R² (day49 holdout — honest, matches LB)'.ljust(63)+'║')
for n, pv in zip(names4, S_val.T):
    print(f'║   {n:<12}: {r2_score(y49,pv):.5f}'.ljust(63)+'║')
print(f'║   Ensemble  : {best_r2_e:.5f}  → score {100*best_r2_e:.3f}'.ljust(63)+'║')
print('╠' + '═'*62 + '╣')
print('║ KEY CHANGES VS V1'.ljust(63)+'║')
changes = [
    'Cross-day TE (day48 stats) replaces leaky random-fold TE',
    'yesterday_demand: day48 (geohash,slot) mean — strongest new feature',
    'DemandNet: geohash embeddings (64-dim) + residual MLP',
    'Temporal split validation (day48→day49) — honest CV',
    'GBM early stopping on day49 val → correct n_estimators',
]
for c in changes:
    print(f'║  • {c}'.ljust(63)+'║')
print('╠' + '═'*62 + '╣')
print('║ SUBMISSION'.ljust(63)+'║')
print(f'║  File  : submissions/final_submission_v3.csv'.ljust(63)+'║')
print(f'║  Shape : {sub_v3.shape[0]} × {sub_v3.shape[1]}'.ljust(63)+'║')
print(f'║  Nulls : {sub_v3.isnull().sum().sum()}'.ljust(63)+'║')
print('╚' + '═'*62 + '╝')

print('\nFirst 10 rows:')
print(sub_v3.head(10).to_string(index=False))
print('\nStats:')
print(sub_v3['demand'].describe().to_string())